<a href="https://colab.research.google.com/github/janvi-pandya/phonepe-transaction-insights/blob/main/Janvi_Pandya_PhonePe_Transaction_Insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - PhonePe Transaction Insights



##### **Project Type**    - EDA + Unsupervised
##### **Contribution**    - Individual
##### **Full Name**       - Janvi Pandya

# **Project Summary -**



```
This project focuses on building an end-to-end data analytics and business intelligence platform using PhonePe transaction data to
analyze digital payment trends, user engagement, insurance adoption, and geographical transaction behavior across India. The system is
designed to simulate a real-world fintech analytics workflow by integrating data extraction, SQL analysis, visualization, and
interactive dashboarding into a single portable solution.

The project begins by extracting raw JSON data from the official GitHub repository provided in the dataset reference. Using
Python-based ETL processes, the data is cleaned, transformed, and loaded into an SQLite relational database to ensure the project can
run seamlessly on any system without requiring additional database installation or configuration.

Structured tables are created for aggregated, map, and top-level transaction, user, and insurance datasets, enabling efficient
SQL-based analysis. Advanced SQL queries are performed to identify top-performing states and districts, analyze payment category trends,
evaluate user engagement patterns, monitor insurance growth, and study quarterly and yearly transaction performance.

Exploratory Data Analysis (EDA) is conducted using Python libraries such as Pandas, Plotly, Matplotlib, and Seaborn to generate
interactive visualizations including bar charts, line graphs, pie charts, heatmaps, and geographical insights. An interactive Streamlit
dashboard is developed to provide real-time analytical exploration through filters, KPIs, and dynamic charts for enhanced business
understanding.

To further strengthen the analytical capabilities of the project, unsupervised machine learning techniques such as K-Means Clustering
are applied to perform regional and transaction-based segmentation. This helps identify high-growth digital payment regions, emerging
markets, and user behavior patterns that can support strategic business decisions and targeted marketing initiatives.

The project demonstrates practical skills in ETL pipeline development, SQL database management, data analysis, visualization,
dashboard creation, and analytical problem-solving while delivering actionable insights for digital payment ecosystems and financial
technology applications.
```



# **GitHub Link -**

https://github.com/janvi-pandya/phonepe-transaction-insights

# **Problem Statement**


**With the increasing reliance on digital payment systems like PhonePe,understanding the dynamics of transactions, user engagement,
and insurance-related data is crucial for improving services and targeting users effectively. This project aims to analyze and
visualize aggregated values of payment categories, create maps for total values at state and district levels, and identify
top-performing states, districts, and pin codes.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import os
import json
import glob
import sqlite3

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sqlalchemy import create_engine

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

### Dataset Loading

In [ ]:
!git clone https://github.com/PhonePe/pulse

### Dataset First View

In [ ]:
# Dataset First Look
!ls -R pulse

### Data Extraction - Aggregated Transactions

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Aggregated_transaction'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure of aggregated_transaction_df
# Columns: State, Year, Quarter, Transaction_Type, Transaction_Count, Transaction_Amount
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Transaction_Type TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

agg_trans_path = 'pulse/data/aggregated/transaction/country/india/state/'
agg_trans_list = os.listdir(agg_trans_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in agg_trans_list:
    cur_state = agg_trans_path + state + '/'
    agg_year_list = os.listdir(cur_state)

    for year in agg_year_list:
        cur_year = cur_state + year + '/'
        agg_file_list = os.listdir(cur_year)

        for file in agg_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            A = json.load(data)

            for i in A['data']['transactionData']:
                name = i['name']
                count = i['paymentInstruments'][0]['count']
                amount = i['paymentInstruments'][0]['amount']

                # Collect the data for insertion
                insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Transaction_Type, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
example_df_from_db = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(example_df_from_db)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

# The original `aggregated_transaction_df` DataFrame creation is no longer needed
# aggregated_transaction_df = pd.DataFrame(columns)
# aggregated_transaction_df.head()


### Data Extraction - Aggregated Users

In [ ]:
# Define database name (assuming it's the same as used for Aggregated_transaction)
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Aggregated_user'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure of aggregated_user_df
# Columns: State, Year, Quarter, Brand, Count, Percentage
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Brand TEXT,
    Count INTEGER,
    Percentage REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

agg_user_path = 'pulse/data/aggregated/user/country/india/state/'
agg_user_list = os.listdir(agg_user_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in agg_user_list:
    cur_state = agg_user_path + state + '/'
    agg_year_list = os.listdir(cur_state)

    for year in agg_year_list:
        cur_year = cur_state + year + '/'
        agg_file_list = os.listdir(cur_year)

        for file in agg_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            B = json.load(data)
            try:
                for i in B['data']['usersByDevice']:
                    brand = i['brand']
                    count = i['count']
                    percentage = i['percentage']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), brand, count, percentage))
            except:
                # Some files might not have 'usersByDevice' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Brand, Count, Percentage)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
aggregated_user_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(aggregated_user_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Data Extraction - Aggregated Insurance

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Aggregated_insurance'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure (State, Year, Quarter, Transaction_Type, Transaction_Count, Transaction_Amount)
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Transaction_Type TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

agg_insur_path = 'pulse/data/aggregated/insurance/country/india/state/'
agg_insur_list = os.listdir(agg_insur_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in agg_insur_list:
    cur_state = agg_insur_path + state + '/'
    agg_year_list = os.listdir(cur_state)

    for year in agg_year_list:
        cur_year = cur_state + year + '/'
        agg_file_list = os.listdir(cur_year)

        for file in agg_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            C = json.load(data)

            try:
                for i in C['data']['transactionData']:
                    name = i['name']
                    count = i['paymentInstruments'][0]['count']
                    amount = i['paymentInstruments'][0]['amount']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))
            except:
                # Some files might not have 'transactionData' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Transaction_Type, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
aggregated_insurance_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(aggregated_insurance_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Data Extraction - Map Transactions

In [ ]:
# Define database name (assuming it's the same as used for Aggregated_transaction)
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Map_transaction'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure of map_transaction_df
# Columns: State, Year, Quarter, District, Transaction_Count, Transaction_Amount
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    District TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

map_trans_path = 'pulse/data/map/transaction/hover/country/india/state/'
map_trans_list = os.listdir(map_trans_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in map_trans_list:
    cur_state = map_trans_path + state + '/'
    map_year_list = os.listdir(cur_state)

    for year in map_year_list:
        cur_year = cur_state + year + '/'
        map_file_list = os.listdir(cur_year)

        for file in map_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            D = json.load(data)

            try:
                for i in D['data']['hoverDataList']:
                    name = i['name']
                    count = i['metric'][0]['count']
                    amount = i['metric'][0]['amount']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))
            except KeyError:
                # Some files might not have 'hoverDataList' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, District, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
map_transaction_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(map_transaction_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Data Extraction - Map Users

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Map_user'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure of map_user_df
# Columns: State, Year, Quarter, District, Registered_Users, App_Opens
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    District TEXT,
    Registered_Users INTEGER,
    App_Opens INTEGER
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

map_user_path = 'pulse/data/map/user/hover/country/india/state/'
map_user_list = os.listdir(map_user_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in map_user_list:
    cur_state = map_user_path + state + '/'
    map_year_list = os.listdir(cur_state)

    for year in map_year_list:
        cur_year = cur_state + year + '/'
        map_file_list = os.listdir(cur_year)

        for file in map_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            E = json.load(data)

            try:
                if 'hoverDataList' in E['data']:
                    for i in E['data']['hoverDataList']:
                        name = i['name']
                        registered_users = i['metric'][0]['registeredUsers']
                        app_opens = i['metric'][0]['appOpens']

                        # Collect the data for insertion
                        insert_records.append((state, int(year), int(file.strip('.json')), name, registered_users, app_opens))
                elif 'hoverData' in E['data'] and E['data']['hoverData'] is not None:
                    # Handle cases where 'hoverData' is a dictionary of districts
                    for district_name, metrics_dict in E['data']['hoverData'].items():
                        registered_users = metrics_dict.get('registeredUsers')
                        app_opens = metrics_dict.get('appOpens')

                        if registered_users is not None and app_opens is not None:
                            insert_records.append((state, int(year), int(file.strip('.json')), district_name, registered_users, app_opens))
                        else:
                            print(f"Warning: Missing 'registeredUsers' or 'appOpens' in hoverData for district {district_name} in {cur_file}. Skipping this entry.")
                else:
                    print(f"Warning: Neither 'hoverDataList' nor 'hoverData' found in {cur_file}. Skipping this file.")
            except KeyError as e:
                print(f"Error processing {cur_file}: {e}. Skipping this file.")

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, District, Registered_Users, App_Opens)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
map_user_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(map_user_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Data Extraction - Map Insurance

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Map_insurance'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    District TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

map_insur_path = 'pulse/data/map/insurance/hover/country/india/state/'
map_insur_list = os.listdir(map_insur_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in map_insur_list:
    cur_state = map_insur_path + state + '/'
    map_year_list = os.listdir(cur_state)

    for year in map_year_list:
        cur_year = cur_state + year + '/'
        map_file_list = os.listdir(cur_year)

        for file in map_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            F = json.load(data)

            try:
                for i in F['data']['hoverDataList']:
                    name = i['name']
                    count = i['metric'][0]['count']
                    amount = i['metric'][0]['amount']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))
            except KeyError:
                # Some files might not have 'hoverDataList' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, District, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
map_insurance_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(map_insurance_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Data Extraction - Top Transactions

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Top_transaction'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Pincode TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

top_trans_path = 'pulse/data/top/transaction/country/india/state/'
top_trans_list = os.listdir(top_trans_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in top_trans_list:
    cur_state = top_trans_path + state + '/'
    top_year_list = os.listdir(cur_state)

    for year in top_year_list:
        cur_year = cur_state + year + '/'
        top_file_list = os.listdir(cur_year)

        for file in top_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            G = json.load(data)

            try:
                for i in G['data']['pincodes']:
                    name = i['entityName']
                    count = i['metric']['count']
                    amount = i['metric']['amount']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))
            except KeyError:
                # Some files might not have 'pincodes' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Pincode, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
top_transaction_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(top_transaction_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

# The original `top_transaction_df` DataFrame creation is no longer needed
# top_transaction_df = pd.DataFrame(columns)
# top_transaction_df.head()

### Data Extraction - Top Users

In [ ]:
# Define database name
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Top_user'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement based on the structure (State, Year, Quarter, Pincode, Registered_Users)
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Pincode TEXT,
    Registered_Users INTEGER
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

top_user_path = 'pulse/data/top/user/country/india/state/'
top_user_list = os.listdir(top_user_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in top_user_list:
    cur_state = top_user_path + state + '/'
    top_year_list = os.listdir(cur_state)

    for year in top_year_list:
        cur_year = cur_state + year + '/'
        top_file_list = os.listdir(cur_year)

        for file in top_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            H = json.load(data)

            try:
                for i in H['data']['pincodes']:
                    try:
                        pincode = i['name']
                        registered_users = i['registeredUsers']

                        # Collect the data for insertion
                        insert_records.append((state, int(year), int(file.strip('.json')), pincode, registered_users))
                    except KeyError as e:
                        print(f"Warning: Missing key in pincode data for {cur_file}: {e}. Skipping this entry.")
                        pass # Skip this entry if keys are missing
            except KeyError:
                print(f"Warning: 'pincodes' key not found in {cur_file}. Skipping this file.")
                pass # Skip this file if 'pincodes' is missing

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Pincode, Registered_Users)
VALUES (?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
top_user_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(top_user_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

# The original `top_user_df` DataFrame creation is no longer needed
# top_user_df = pd.DataFrame(columns)
# top_user_df.head()

### Data Extraction - Top Insurance

In [ ]:
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Define table name
table_name = 'Top_insurance'

# Drop table if it exists to ensure a clean slate
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create table statement
create_table_sql = f"""
CREATE TABLE {table_name} (
    State TEXT,
    Year INTEGER,
    Quarter INTEGER,
    Pincode TEXT,
    Transaction_Count INTEGER,
    Transaction_Amount REAL
)
"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created with SQL: {create_table_sql}")

top_insur_path = 'pulse/data/top/insurance/country/india/state/'
top_insur_list = os.listdir(top_insur_path)

# Prepare for batch insertion for efficiency
insert_records = []

for state in top_insur_list:
    cur_state = top_insur_path + state + '/'
    top_year_list = os.listdir(cur_state)

    for year in top_year_list:
        cur_year = cur_state + year + '/'
        top_file_list = os.listdir(cur_year)

        for file in top_file_list:
            cur_file = cur_year + file
            data = open(cur_file, 'r')
            I = json.load(data)

            try:
                for i in I['data']['pincodes']:
                    name = i['entityName']
                    count = i['metric']['count']
                    amount = i['metric']['amount']

                    # Collect the data for insertion
                    insert_records.append((state, int(year), int(file.strip('.json')), name, count, amount))
            except KeyError:
                # Some files might not have 'pincodes' or other expected keys
                pass

# Insert all collected records in a batch
insert_sql = f"""
INSERT INTO {table_name} (State, Year, Quarter, Pincode, Transaction_Count, Transaction_Amount)
VALUES (?, ?, ?, ?, ?, ?)
"""
cursor.executemany(insert_sql, insert_records)
conn.commit()
print(f"Inserted {len(insert_records)} records into '{table_name}'.")

# Verify data insertion (optional, but good for confirmation)
print(f"\nVerifying data for '{table_name}':")
top_insurance_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
display(top_insurance_df)

conn.close()
print(f"Connection to '{DB_NAME}' closed.")

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

In [ ]:
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)

# Re-load dataframes from the database
aggregated_transaction_df = pd.read_sql_query("SELECT * FROM Aggregated_transaction", conn)
aggregated_user_df = pd.read_sql_query("SELECT * FROM Aggregated_user", conn)
aggregated_insurance_df = pd.read_sql_query("SELECT * FROM Aggregated_insurance", conn)
map_transaction_df = pd.read_sql_query("SELECT * FROM Map_transaction", conn)
map_user_df = pd.read_sql_query("SELECT * FROM Map_user", conn)
map_insurance_df = pd.read_sql_query("SELECT * FROM Map_insurance", conn)
top_transaction_df = pd.read_sql_query("SELECT * FROM Top_transaction", conn)
top_user_df = pd.read_sql_query("SELECT * FROM Top_user", conn)
top_insurance_df = pd.read_sql_query("SELECT * FROM Top_insurance", conn)

conn.close()

print(f"aggregated_transaction_df: {aggregated_transaction_df.shape[0]} rows, {aggregated_transaction_df.shape[1]} columns")
print(f"aggregated_user_df: {aggregated_user_df.shape[0]} rows, {aggregated_user_df.shape[1]} columns")
print(f"aggregated_insurance_df: {aggregated_insurance_df.shape[0]} rows, {aggregated_insurance_df.shape[1]} columns")
print(f"map_transaction_df: {map_transaction_df.shape[0]} rows, {map_transaction_df.shape[1]} columns")
print(f"map_user_df: {map_user_df.shape[0]} rows, {map_user_df.shape[1]} columns")
print(f"map_insurance_df: {map_insurance_df.shape[0]} rows, {map_insurance_df.shape[1]} columns")
print(f"top_transaction_df: {top_transaction_df.shape[0]} rows, {top_transaction_df.shape[1]} columns")
print(f"top_user_df: {top_user_df.shape[0]} rows, {top_user_df.shape[1]} columns")
print(f"top_insurance_df: {top_insurance_df.shape[0]} rows, {top_insurance_df.shape[1]} columns")

### Dataset Information

In [ ]:
# Dataset Info

In [ ]:
DB_NAME = 'phonepe_pulse.db'
conn = sqlite3.connect(DB_NAME)

# Re-load dataframes from the database
aggregated_transaction_df = pd.read_sql_query("SELECT * FROM Aggregated_transaction", conn)
aggregated_user_df = pd.read_sql_query("SELECT * FROM Aggregated_user", conn)
aggregated_insurance_df = pd.read_sql_query("SELECT * FROM Aggregated_insurance", conn)
map_transaction_df = pd.read_sql_query("SELECT * FROM Map_transaction", conn)
map_user_df = pd.read_sql_query("SELECT * FROM Map_user", conn)
map_insurance_df = pd.read_sql_query("SELECT * FROM Map_insurance", conn)
top_transaction_df = pd.read_sql_query("SELECT * FROM Top_transaction", conn)
top_user_df = pd.read_sql_query("SELECT * FROM Top_user", conn)
top_insurance_df = pd.read_sql_query("SELECT * FROM Top_insurance", conn)

conn.close()

print('aggregated_transaction_df info:')
aggregated_transaction_df.info()
print('\naggregated_user_df info:')
aggregated_user_df.info()
print('\naggregated_insurance_df info:')
aggregated_insurance_df.info()
print('\nmap_transaction_df info:')
map_transaction_df.info()
print('\nmap_user_df info:')
map_user_df.info()
print('\nmap_insurance_df info:')
map_insurance_df.info()
print('\ntop_transaction_df info:')
top_transaction_df.info()
print('\ntop_user_df info:')
top_user_df.info()
print('\ntop_insurance_df info:')
top_insurance_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count

In [ ]:
print(f"aggregated_transaction_df duplicates: {aggregated_transaction_df.duplicated().sum()}")
print(f"aggregated_user_df duplicates: {aggregated_user_df.duplicated().sum()}")
print(f"aggregated_insurance_df duplicates: {aggregated_insurance_df.duplicated().sum()}")
print(f"map_transaction_df duplicates: {map_transaction_df.duplicated().sum()}")
print(f"map_user_df duplicates: {map_user_df.duplicated().sum()}")
print(f"map_insurance_df duplicates: {map_insurance_df.duplicated().sum()}")
print(f"top_transaction_df duplicates: {top_transaction_df.duplicated().sum()}")
print(f"top_user_df duplicates: {top_user_df.duplicated().sum()}")
print(f"top_insurance_df duplicates: {top_insurance_df.duplicated().sum()}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count

In [ ]:
print('Missing values in aggregated_transaction_df:')
display(aggregated_transaction_df.isnull().sum())
print('\nMissing values in aggregated_user_df:')
display(aggregated_user_df.isnull().sum())
print('\nMissing values in aggregated_insurance_df:')
display(aggregated_insurance_df.isnull().sum())
print('\nMissing values in map_transaction_df:')
display(map_transaction_df.isnull().sum())
print('\nMissing values in map_user_df:')
display(map_user_df.isnull().sum())
print('\nMissing values in map_insurance_df:')
display(map_insurance_df.isnull().sum())
print('\nMissing values in top_transaction_df:')
display(top_transaction_df.isnull().sum())
print('\nMissing values in top_user_df:')
display(top_user_df.isnull().sum())
print('\nMissing values in top_insurance_df:')
display(top_insurance_df.isnull().sum())

In [ ]:
dataframes = {
    'aggregated_transaction_df': aggregated_transaction_df,
    'aggregated_user_df': aggregated_user_df,
    'aggregated_insurance_df': aggregated_insurance_df,
    'map_transaction_df': map_transaction_df,
    'map_user_df': map_user_df,
    'map_insurance_df': map_insurance_df,
    'top_transaction_df': top_transaction_df,
    'top_user_df': top_user_df,
    'top_insurance_df': top_insurance_df
}

for name, df in dataframes.items():
    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(10, 6))
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
        plt.title(f'Missing Values in {name}')
        plt.show()
    else:
        print(f'No missing values found in {name}.')


### What did you know about your dataset?

### Dataset Overview:

We have successfully extracted and loaded data into nine Pandas DataFrames, categorized as aggregated, map, and top-level data for transactions, users, and insurance. The data spans from 2018 to 2024 (though some categories may have fewer years).

### Key Findings:

*   **Dimensions:**
    *   `aggregated_transaction_df`: 5034 rows, 6 columns
    *   `aggregated_user_df`: 6732 rows, 6 columns
    *   `aggregated_insurance_df`: 682 rows, 6 columns
    *   `map_transaction_df`: 20604 rows, 6 columns
    *   `map_user_df`: 20608 rows, 6 columns
    *   `map_insurance_df`: 13876 rows, 6 columns
    *   `top_transaction_df`: 9999 rows, 6 columns
    *   `top_user_df`: 10000 rows, 5 columns
    *   `top_insurance_df`: 6668 rows, 6 columns

*   **Data Types:** All DataFrames have appropriate data types, including `object` for categorical fields like `State` and `Transaction_Type`, `int64` for `Year`, `Quarter`, and counts, and `float64` for amounts and percentages.

*   **Duplicate Values:** No duplicate rows were found across any of the nine DataFrames, indicating a clean dataset in terms of exact row duplicates.

*   **Missing Values:**
    *   `aggregated_transaction_df`, `aggregated_user_df`, `aggregated_insurance_df`, `map_transaction_df`, `map_user_df`, `map_insurance_df`, and `top_user_df` have no missing values.
    *   `top_transaction_df` has 2 missing values in the 'Pincode' column.
    *   `top_insurance_df` has 3 missing values in the 'Pincode' column.

These missing values in the 'Pincode' column for `top_transaction_df` and `top_insurance_df` will need to be addressed during the data wrangling phase, possibly by imputation or removal, depending on their impact.

## ***2. Understanding Your Variables***

In [ ]:
print('aggregated_transaction_df columns:')
display(aggregated_transaction_df.columns)
print('\naggregated_user_df columns:')
display(aggregated_user_df.columns)
print('\naggregated_insurance_df columns:')
display(aggregated_insurance_df.columns)
print('\nmap_transaction_df columns:')
display(map_transaction_df.columns)
print('\nmap_user_df columns:')
display(map_user_df.columns)
print('\nmap_insurance_df columns:')
display(map_insurance_df.columns)
print('\ntop_transaction_df columns:')
display(top_transaction_df.columns)
print('\ntop_user_df columns:')
display(top_user_df.columns)
print('\ntop_insurance_df columns:')
display(top_insurance_df.columns)

In [ ]:
# Dataset Describe

In [ ]:
print('aggregated_transaction_df describe:')
display(aggregated_transaction_df.describe())
print('\naggregated_user_df describe:')
display(aggregated_user_df.describe())
print('\naggregated_insurance_df describe:')
display(aggregated_insurance_df.describe())
print('\nmap_transaction_df describe:')
display(map_transaction_df.describe())
print('\nmap_user_df describe:')
display(map_user_df.describe())
print('\nmap_insurance_df describe:')
display(map_insurance_df.describe())
print('\ntop_transaction_df describe:')
display(top_transaction_df.describe())
print('\ntop_user_df describe:')
display(top_user_df.describe())
print('\ntop_insurance_df describe:')
display(top_insurance_df.describe())

### Variables Description

#### **1. `aggregated_transaction_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the transaction data.
- **Quarter**: `int64`, Quarter of the year (1, 2, 3, or 4).
- **Transaction_Type**: `object`, Category of transaction (e.g., Peer-to-peer payments, Merchant payments).
- **Transaction_Count**: `int64`, Number of transactions for the given type.
- **Transaction_Amount**: `float64`, Total amount of transactions for the given type.

#### **2. `aggregated_user_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the user data.
- **Quarter**: `int64`, Quarter of the year.
- **Brand**: `object`, Mobile phone brand used by users.
- **Count**: `int64`, Number of users for that brand.
- **Percentage**: `float64`, Percentage of users for that brand within the state and quarter.

#### **3. `aggregated_insurance_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the insurance data.
- **Quarter**: `int64`, Quarter of the year.
- **Transaction_Type**: `object`, Type of insurance transaction.
- **Transaction_Count**: `int64`, Number of insurance transactions.
- **Transaction_Amount**: `float64`, Total amount of insurance transactions.

#### **4. `map_transaction_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the transaction data.
- **Quarter**: `int64`, Quarter of the year.
- **District**: `object`, Name of the district.
- **Transaction_Count**: `int64`, Number of transactions in the district.
- **Transaction_Amount**: `float64`, Total amount of transactions in the district.

#### **5. `map_user_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the user data.
- **Quarter**: `int64`, Quarter of the year.
- **District**: `object`, Name of the district.
- **Registered_Users**: `int64`, Number of registered PhonePe users in the district.
- **App_Opens**: `int64`, Number of times the PhonePe app was opened in the district.

#### **6. `map_insurance_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the insurance data.
- **Quarter**: `int64`, Quarter of the year.
- **District**: `object`, Name of the district.
- **Transaction_Count**: `int64`, Number of insurance transactions in the district.
- **Transaction_Amount**: `float64`, Total amount of insurance transactions in the district.

#### **7. `top_transaction_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the transaction data.
- **Quarter**: `int64`, Quarter of the year.
- **Pincode**: `object`, Pincode of the location.
- **Transaction_Count**: `int64`, Number of transactions in the pincode.
- **Transaction_Amount**: `float64`, Total amount of transactions in the pincode.

#### **8. `top_user_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the user data.
- **Quarter**: `int64`, Quarter of the year.
- **Pincode**: `object`, Pincode of the location.
- **Registered_Users**: `int64`, Number of registered PhonePe users in the pincode.

#### **9. `top_insurance_df`**
- **State**: `object`, Name of the Indian state.
- **Year**: `int64`, Year of the insurance data.
- **Quarter**: `int64`, Quarter of the year.
- **Pincode**: `object`, Pincode of the location.
- **Transaction_Count**: `int64`, Number of insurance transactions in the pincode.
- **Transaction_Amount**: `float64`, Total amount of insurance transactions in the pincode.

### Check Unique Values for each variable.

In [ ]:
for df_name, df in dataframes.items():
    print(f'\nUnique values for {df_name}:')
    for column in df.columns:
        print(f"  - {column}: {df[column].nunique()} unique values")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Drop rows with missing 'Pincode' values in top_transaction_df
top_transaction_df.dropna(subset=['Pincode'], inplace=True)

# Drop rows with missing 'Pincode' values in top_insurance_df
top_insurance_df.dropna(subset=['Pincode'], inplace=True)

print('Missing values after handling in top_transaction_df:')
display(top_transaction_df.isnull().sum())
print('\nMissing values after handling in top_insurance_df:')
display(top_insurance_df.isnull().sum())

print(f"\nUpdated top_transaction_df: {top_transaction_df.shape[0]} rows, {top_transaction_df.shape[1]} columns")
print(f"Updated top_insurance_df: {top_insurance_df.shape[0]} rows, {top_insurance_df.shape[1]} columns")

### What all manipulations have you done and insights you found?

### Manipulations Performed:
1.  **Handling Missing Values in 'Pincode' column:** I identified and subsequently dropped rows with missing values in the 'Pincode' column for both `top_transaction_df` and `top_insurance_df`.

### Insights Found:
*   **Minimal Data Loss:** The number of missing 'Pincode' values was very small (2 in `top_transaction_df` and 3 in `top_insurance_df`). This means dropping these rows resulted in minimal data loss and did not significantly alter the overall size or representativeness of the datasets.
*   **Improved Data Quality:** By removing these incomplete entries, we've ensured that any future analysis involving pincode-level data will be based on clean and complete records. This is crucial for accurate geographical and granular insights, especially when identifying top-performing areas.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code

# Group by Year and Quarter to get total transaction amount and count
transaction_trend = aggregated_transaction_df.groupby(['Year', 'Quarter']).agg(
    Total_Transaction_Amount=('Transaction_Amount', 'sum'),
    Total_Transaction_Count=('Transaction_Count', 'sum')
).reset_index()

# Create a combined 'Year-Quarter' column for better plotting on x-axis
transaction_trend['Year_Quarter'] = transaction_trend['Year'].astype(str) + '-Q' + transaction_trend['Quarter'].astype(str)

# Plotting Total Transaction Amount over time
fig = px.line(
    transaction_trend,
    x='Year_Quarter',
    y='Total_Transaction_Amount',
    title='Overall Transaction Amount Trend Across Years and Quarters',
    labels={'Total_Transaction_Amount': 'Total Transaction Amount (INR)', 'Year_Quarter': 'Year and Quarter'},
    markers=True,
    hover_data={'Total_Transaction_Count': True, 'Total_Transaction_Amount': ':.2f'}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

I chose a line chart for the 'Overall Transaction Amount Trend Across Years and Quarters' because:

*   **Time-Series Data:** Line charts are ideal for displaying data that changes continuously over time. Our data spans multiple years and quarters, making a line chart the most effective way to show the progression and trends of transaction amounts.
*   **Trend Identification:** The primary goal was to observe the overall growth or decline and identify any recurring patterns or fluctuations. A line chart clearly highlights these trends and makes it easy to spot upward or downward movements.
*   **Comparison over Time:** It allows for easy comparison of transaction amounts across different quarters and years, helping to understand both short-term variations and long-term growth trajectories.
*   **Clarity and Readability:** When dealing with continuous data like transaction amounts over regular time intervals, a line chart provides a clear and uncluttered visual representation, making it easy for viewers to grasp the information quickly.

##### 2. What is/are the insight(s) found from the chart?

The 'Overall Transaction Amount Trend Across Years and Quarters' chart reveals several key insights:

*   **Consistent Growth:** There is a clear and consistent upward trend in the total transaction amount over the years, indicating robust growth in digital payments through PhonePe.
*   **Quarterly Fluctuations:** While the overall trend is upward, there are noticeable quarterly fluctuations. Typically, transaction amounts tend to increase towards the end of the year (Q3 and Q4) and might dip slightly in Q1 or Q2, possibly aligning with seasonal spending patterns or festive periods.
*   **Accelerated Growth in Recent Years:** The steepness of the upward curve appears to increase in more recent years (e.g., from 2021 onwards), suggesting an acceleration in the adoption and usage of PhonePe's transaction services.
*   **Magnitude of Growth:** The transaction amounts have grown from hundreds of billions to trillions of INR, demonstrating a significant expansion in market penetration and user activity.
*   **Data Completeness:** The data appears to be consistent across years and quarters, providing a reliable basis for trend analysis.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights gained from the 'Overall Transaction Amount Trend Across Years and Quarters' chart can definitely create a positive business impact. Here's how:

**Positive Business Impact:**
1.  **Strategic Investment & Expansion:** The consistent upward trend and accelerated growth in recent years signal a healthy and expanding market. This insight supports decisions for further investment in infrastructure, marketing, and product development to capture more market share. It can justify expanding into new regions or introducing new features.
2.  **Revenue Forecasting & Budgeting:** Understanding the growth trajectory and seasonal fluctuations allows for more accurate revenue forecasting. Businesses can set realistic targets, allocate resources effectively, and prepare for peak transaction periods (e.g., Q3/Q4) with adequate support and promotional campaigns.
3.  **Performance Benchmarking:** The trend serves as a benchmark for evaluating the effectiveness of business strategies. If transaction growth aligns with or exceeds the observed trend, it indicates successful initiatives. Conversely, a slowdown against the trend could flag issues that need investigation.
4.  **Investor Confidence:** A clear growth story, backed by data, is crucial for attracting and retaining investors. It demonstrates the viability and potential of the platform.
5.  **Optimized Marketing & Promotions:** By understanding quarterly fluctuations, marketing teams can strategically plan campaigns to either capitalize on peak periods or stimulate growth during slower quarters. For instance, targeted promotions during Q1/Q2 might help mitigate natural dips.

**Insights Leading to Negative Growth (or areas of concern):**
Currently, the chart primarily shows **positive growth** with no overall negative growth trend. However, the 'negative' aspects are more accurately described as **areas for optimization or potential risks if not managed**:

*   **Quarterly Dips (e.g., Q1/Q2):** While not 'negative growth' in the overall sense, the *relative slowdown* or slight dip in transaction amounts during certain quarters (e.g., typically Q1 or Q2 compared to Q4) indicates a seasonal pattern. If these dips become more pronounced or prolonged, it could signal issues like decreased user engagement during specific times of the year, increased competition, or ineffective seasonal campaigns.
    *   **Justification:** These fluctuations, if not understood and addressed, could lead to missed opportunities for growth during those periods. For instance, if a Q1 dip is larger than expected, it might suggest that post-holiday spending habits or other factors are impacting user behavior more significantly than anticipated, requiring a strategic response to re-engage users or offer incentives.

In summary, the dominant insight is strong positive growth. The quarterly fluctuations present opportunities for targeted strategies rather than signaling immediate negative growth, but they are crucial to monitor to prevent future negative trends.

#### Chart - 2

In [ ]:
transaction_type_trend = aggregated_transaction_df.groupby(['Year', 'Quarter', 'Transaction_Type']).agg(
    Total_Transaction_Amount=('Transaction_Amount', 'sum')
).reset_index()

transaction_type_trend['Year_Quarter'] = transaction_type_trend['Year'].astype(str) + '-Q' + transaction_type_trend['Quarter'].astype(str)

fig = px.line(
    transaction_type_trend,
    x='Year_Quarter',
    y='Total_Transaction_Amount',
    color='Transaction_Type',
    title='Transaction Amount Trend by Transaction Type Across Years and Quarters',
    labels={
        'Total_Transaction_Amount': 'Total Transaction Amount (INR)',
        'Year_Quarter': 'Year and Quarter',
        'Transaction_Type': 'Transaction Type'
    },
    markers=True,
    hover_name='Transaction_Type'
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

I chose a multi-line chart for the 'Transaction Amount Trend by Transaction Type Across Years and Quarters' because:

*   **Comparison of Multiple Categories Over Time:** A multi-line chart excels at comparing the trends of several different categories (in this case, `Transaction_Type`s) on the same time-series axis. This allows for easy visual identification of which transaction types are growing, declining, or remaining stable relative to each other.
*   **Detailed Trend Identification:** While Chart 1 showed the overall trend, this chart dives deeper, revealing individual growth patterns, seasonality, and significant events specific to each transaction type. For example, it can highlight if 'Merchant payments' are growing faster than 'Peer-to-peer payments' or if certain types experience more pronounced quarterly dips.
*   **Highlighting Divergence/Convergence:** It clearly shows if the various transaction types are moving in parallel, diverging (e.g., one type growing rapidly while another stagnates), or converging over time, providing crucial context for strategic decisions.
*   **Clarity for Categorical Time-Series Data:** For continuous data (transaction amount) across discrete time intervals (Year-Quarter) and distinct categories (Transaction_Type), the use of different colored lines for each category maintains clarity and prevents visual clutter, making the chart interpretable.

##### 2. What is/are the insight(s) found from the chart?

The 'Transaction Amount Trend by Transaction Type Across Years and Quarters' chart provides granular insights into the performance of different transaction categories:

*   **Dominance of Peer-to-peer payments:** 'Peer-to-peer payments' consistently hold the largest share of the total transaction amount and show robust growth over the entire period, reaffirming its foundational role in PhonePe's ecosystem.
*   **Strong Growth in Merchant payments:** 'Merchant payments' also exhibit significant and consistent growth, particularly from 2020 onwards, indicating increasing adoption of PhonePe for commercial transactions. This segment is crucial for the platform's long-term revenue and ecosystem expansion.
*   **Steady but Slower Growth in Recharge & bill payments:** 'Recharge & bill payments' show a steady upward trend, but the growth rate appears less aggressive compared to 'Peer-to-peer' and 'Merchant payments'. This category likely represents a stable utility service rather than a high-growth area.
*   **Relatively Smaller but Growing 'Financial Services' and 'Others':** 'Financial Services' and 'Others' transaction types represent a much smaller portion of the total transaction amount. However, both show a gradual but noticeable increase over time, suggesting emerging opportunities or diversification.
*   **Seasonal Patterns Across Types:** Similar to the overall trend, most transaction types show some quarterly fluctuations, often peaking in later quarters (Q3 and Q4) and slightly dipping in earlier ones. This seasonality is inherent across most transaction categories.
*   **Shift in Composition:** While 'Peer-to-peer' remains dominant, the growth rate of 'Merchant payments' suggests a gradual shift towards a more balanced transaction portfolio, where commercial transactions are gaining more prominence relative to personal transfers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The insights from the 'Transaction Amount Trend by Transaction Type Across Years and Quarters' chart offer significant opportunities for positive business impact, with some areas requiring careful monitoring to avoid potential negative implications.

**Positive Business Impact:**

1.  **Strategic Focus on High-Growth Segments:**
    *   **Insight:** The strong and continuous growth in 'Peer-to-peer payments' and 'Merchant payments' highlights these as core strengths.
    *   **Impact:** PhonePe can prioritize investments in these areas, such as enhancing user experience, expanding merchant acquisition programs, and offering incentives to further accelerate their growth. Focusing on 'Merchant payments' is especially vital as it often translates to higher revenue per transaction and strengthens the platform's commercial ecosystem.

2.  **Product Development and Diversification:**
    *   **Insight:** The consistent, albeit slower, growth in 'Recharge & bill payments' indicates a reliable, utility-driven user base. The nascent growth in 'Financial Services' and 'Others' suggests potential for diversification.
    *   **Impact:** For 'Recharge & bill payments', focus can be on efficiency and convenience to retain users. For 'Financial Services', the gradual increase indicates a receptive market. PhonePe can invest in developing and promoting new financial products (e.g., loans, investments, insurance products – which might fall under 'Financial Services' or 'Others') to tap into these emerging trends and expand its service offerings beyond basic payments.

3.  **Targeted Marketing and User Engagement:**
    *   **Insight:** Understanding which transaction types are growing fastest and their relative volumes allows for targeted marketing campaigns.
    *   **Impact:** Campaigns can be designed to onboard more merchants, promote specific financial services, or drive usage of less dominant categories to increase overall transaction volume and user stickiness. For instance, promoting merchant payment features in regions with lower adoption.

4.  **Resource Allocation and Capacity Planning:**
    *   **Insight:** Different transaction types might have varying infrastructure and support requirements.
    *   **Impact:** By identifying the dominant and growing types, PhonePe can allocate resources (e.g., server capacity, customer support, fraud detection for specific transaction patterns) more effectively to ensure seamless operations and scalability.

**Insights Leading to Negative Growth (or areas of concern):**

Currently, the chart primarily shows **positive growth** across all transaction types, with no clear indication of negative growth. However, some aspects could become areas of concern if not proactively managed:

*   **Reliance on Peer-to-peer Payments:**
    *   **Insight:** 'Peer-to-peer payments' remain significantly larger than other categories.
    *   **Justification:** While a strength, over-reliance on one transaction type can be a risk. If regulations change, a major competitor emerges, or user preferences shift away from P2P, the overall transaction volume could be significantly impacted. PhonePe needs to continue diversifying its revenue streams and encouraging usage of other categories to mitigate this risk.
*   **Stagnation in Smaller Categories:**
    *   **Insight:** While 'Financial Services' and 'Others' are growing, their absolute volumes are still very low compared to the main categories.
    *   **Justification:** If these smaller categories fail to gain significant traction, it could indicate missed opportunities for market expansion and revenue diversification. It might also signal that existing offerings in these segments are not compelling enough or that the market is saturated/competitive. A lack of substantial growth here could limit PhonePe's ability to capture new market segments or increase its average revenue per user. This could necessitate a re-evaluation of product-market fit or marketing strategies for these specific services.

In summary, the insights primarily point towards a healthy and expanding business with diversified growth drivers. The "negative" aspects are more about strategic risks and areas for proactive intervention rather than immediate declining trends. Addressing these potential concerns will help sustain and enhance the positive business trajectory.

#### Chart - 3

In [ ]:
user_reg_trend = map_user_df.groupby(['Year', 'Quarter']).agg(
    Total_Registered_Users=('Registered_Users', 'sum')
).reset_index()

user_reg_trend['Year_Quarter'] = user_reg_trend['Year'].astype(str) + '-Q' + user_reg_trend['Quarter'].astype(str)

fig = px.line(
    user_reg_trend,
    x='Year_Quarter',
    y='Total_Registered_Users',
    title='Overall Registered Users Trend Across Years and Quarters',
    labels={'Total_Registered_Users': 'Total Registered Users', 'Year_Quarter': 'Year and Quarter'},
    markers=True,
    hover_data={'Total_Registered_Users': True}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

I chose a line chart for the 'Overall Registered Users Trend Across Years and Quarters' because:

*   **Time-Series Data:** A line chart is the most effective way to display data that changes over continuous time, such as the number of registered users across different years and quarters. It clearly illustrates the progression and evolution of the user base.
*   **Trend Identification:** The primary objective is to observe the overall growth trajectory, acceleration, or any plateaus in user registration. A line chart makes it easy to identify these trends and patterns at a glance.
*   **Long-Term vs. Short-Term Changes:** It allows for a clear distinction between long-term growth (across years) and short-term fluctuations (across quarters), providing a comprehensive view of user acquisition.
*   **Clarity and Simplicity:** For a single metric (Total Registered Users) over time, a line chart offers a clear, uncluttered, and easily understandable visual representation.

##### 2. What is/are the insight(s) found from the chart?

The 'Overall Registered Users Trend Across Years and Quarters' chart provides several key insights into PhonePe's user acquisition and growth:

*   **Exponential Growth:** There is a very strong and consistent exponential growth trend in the total number of registered users over the years, indicating successful user acquisition strategies and increasing market penetration.
*   **Significant User Base Expansion:** The number of registered users has grown from tens of millions in 2018 to hundreds of millions in recent years, demonstrating PhonePe's substantial expansion in the Indian digital payments market.
*   **Steady Quarterly Increases:** Within each year, there are consistent increases in registered users from quarter to quarter, with no significant dips, suggesting continuous onboarding of new users.
*   **Accelerated Adoption:** The curve becomes steeper in recent years, particularly from 2020 onwards, which could be attributed to factors like increased digitalization during the pandemic, government initiatives promoting digital payments, or effective marketing campaigns by PhonePe.
*   **Positive User Sentiment:** The sustained growth suggests a positive perception of PhonePe's services and user experience, leading to continued adoption.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Overall Registered Users Trend Across Years and Quarters' chart are highly valuable for creating a positive business impact.

**Positive Business Impact:**
1.  **Validation of Business Strategy:** The consistent and exponential growth in registered users validates PhonePe's overall business strategy, product-market fit, and marketing efforts. This reinforces confidence in current approaches.
2.  **Investment Attraction:** Strong user growth is a critical metric for investors and stakeholders. This chart provides compelling evidence of a rapidly expanding user base, which can attract further investment and support higher valuations.
3.  **Revenue Potential:** A larger user base directly translates to greater potential for transaction volume, which in turn leads to increased revenue through transaction fees, merchant commissions, and cross-selling of other financial products (e.g., insurance, financial services).
4.  **Market Leadership & Network Effects:** Continuous user growth strengthens PhonePe's position as a market leader. This also creates powerful network effects, where more users attract more merchants, and more merchants attract more users, creating a virtuous cycle of growth.
5.  **Product Development & Expansion:** A growing user base signifies a larger addressable market for new features, services, and product offerings. It encourages investment in R&D and market expansion initiatives.
6.  **Strategic Planning:** The clear growth trend allows for more accurate long-term strategic planning, setting ambitious yet realistic growth targets, and allocating resources effectively for scaling infrastructure and support.

**Insights Leading to Negative Growth (or areas of concern):**
Based on the chart, there are **no insights that indicate negative growth** in registered users. The trend is overwhelmingly positive, showing continuous and accelerating growth. However, a business should always be aware of potential risks that could *lead* to a slowdown or reversal of this growth:

*   **Market Saturation:** While not currently visible, as the user base expands significantly, the growth rate might naturally decelerate as the market approaches saturation. PhonePe would need to identify new avenues for growth, such as expanding into new demographics or international markets, or focusing on increasing engagement and transaction frequency among existing users.
*   **Increased Competition:** A rapidly growing market attracts more competitors. While PhonePe has shown strong growth, intensified competition could slow down its user acquisition if not countered with innovative products or aggressive marketing.
*   **Regulatory Changes:** Adverse regulatory changes in the fintech sector could impact user acquisition or retention.
*   **Security Breaches/Trust Issues:** Any significant security breach or loss of user trust could immediately halt and reverse user growth.

In summary, the chart presents a highly positive outlook for PhonePe's user acquisition. The business impact is overwhelmingly positive, providing a strong foundation for future growth and investment. The 'negative growth' considerations are primarily external or future-oriented risks that need continuous monitoring and strategic planning to mitigate, rather than current trends observed in the data.

#### Chart - 4

In [ ]:
app_opens_trend = map_user_df.groupby(['Year', 'Quarter']).agg(
    Total_App_Opens=('App_Opens', 'sum')
).reset_index()

app_opens_trend['Year_Quarter'] = app_opens_trend['Year'].astype(str) + '-Q' + app_opens_trend['Quarter'].astype(str)

fig = px.line(
    app_opens_trend,
    x='Year_Quarter',
    y='Total_App_Opens',
    title='Overall App Opens Trend Across Years and Quarters',
    labels={'Total_App_Opens': 'Total App Opens', 'Year_Quarter': 'Year and Quarter'},
    markers=True,
    hover_data={'Total_App_Opens': True}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

Answer Here.

I chose a line chart for the 'Overall App Opens Trend Across Years and Quarters' because:

*   **Time-Series Data:** A line chart is the most effective way to display data that evolves continuously over time, such as the number of app opens across different years and quarters. It clearly illustrates the progression and overall trend.
*   **Trend Identification:** The primary objective is to observe the overall trajectory, whether there's growth, decline, or stagnation in app usage. A line chart makes it easy to identify these trends and patterns at a glance.
*   **Long-Term vs. Short-Term Changes:** It allows for a clear distinction between long-term growth (across years) and short-term fluctuations (across quarters), providing a comprehensive view of user engagement.
*   **Clarity and Simplicity:** For a single metric (Total App Opens) over time, a line chart offers a clear, uncluttered, and easily understandable visual representation.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

The 'Overall App Opens Trend Across Years and Quarters' chart reveals the following insights:

*   **Strong Growth in App Engagement:** Similar to registered users, there is a clear and consistent upward trend in the total number of app opens, indicating increasing user engagement with the PhonePe platform.
*   **Correlation with Registered Users:** The trend in app opens closely mirrors the trend in registered users, which is expected as more users generally lead to more app usage. This confirms that new users are actively using the application.
*   **Accelerated Growth Post-2020:** The growth in app opens appears to accelerate significantly from 2020 onwards, aligning with the general increase in digital adoption and the growth observed in transaction amounts and registered users. This could be due to increased marketing efforts, new feature releases, or broader market trends.
*   **Potential for Seasonal Peaks:** While generally increasing, there might be slight quarterly variations, which could suggest seasonal usage patterns. Further analysis could explore these patterns in more detail.
*   **Healthy Ecosystem:** The continuous rise in app opens, coupled with transaction and user growth, indicates a healthy and expanding digital payment ecosystem for PhonePe.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Overall App Opens Trend Across Years and Quarters' chart can significantly contribute to positive business impact:

**Positive Business Impact:**
1.  **Validation of User Engagement Strategies:** Consistent growth in app opens validates that PhonePe's strategies for user engagement, product design, and feature relevance are working. This encourages continued investment in these areas.
2.  **Increased Transaction Potential:** Higher app opens directly correlate with more opportunities for users to conduct transactions. This is a leading indicator for future transaction volume and revenue growth.
3.  **Advertising and Monetization Opportunities:** A highly engaged user base with frequent app opens is attractive to advertisers and partners. It opens up avenues for in-app advertising, cross-selling of financial products, and other monetization strategies.
4.  **Resource Allocation:** Understanding peak usage times and growth trajectories helps in capacity planning for servers, customer support, and marketing campaigns, ensuring a smooth user experience even during high demand.
5.  **Product Feature Prioritization:** Analyzing app open trends alongside feature usage can help prioritize product development. Features that drive higher engagement can be further enhanced, and less used features can be re-evaluated.

**Insights Leading to Negative Growth (or areas of concern):**
Currently, the chart shows **no insights indicating negative growth** in app opens. The trend is consistently positive and growing. However, potential future concerns that could lead to a deceleration or reversal of this trend include:

*   **User Churn/Decreased Engagement:** A slowdown or plateau in app opens, despite continued user registration, would signal that users are registering but not actively using the app, or that existing users are disengaging. This would require investigating the causes of churn and re-engagement strategies.
    *   **Justification:** If app opens per user were to decline, it would indicate a weakening of the platform's core utility or increasing competition. This could lead to a decline in transaction volumes, reduced advertising revenue, and an overall negative impact on the business's financial health and market position.
*   **Technical Issues/Poor User Experience:** Any significant performance issues, bugs, or a deteriorating user experience could lead to a sharp drop in app opens.
*   **New Competition:** The entry of highly innovative or disruptive competitors could siphon off user engagement.

In summary, the current trend is highly positive, reinforcing the success of PhonePe's user engagement strategies. The 'negative growth' considerations are primarily proactive monitoring points to ensure the sustained health and growth of the user base and platform usage.

#### Chart - 5

In [ ]:
# Group by Transaction_Type to get total transaction amount across all years and quarters
transaction_type_total = aggregated_transaction_df.groupby('Transaction_Type').agg(
    Total_Transaction_Amount=('Transaction_Amount', 'sum')
).reset_index().sort_values(by='Total_Transaction_Amount', ascending=False)

# Plotting Total Transaction Amount by Transaction Type as a bar chart
fig = px.bar(
    transaction_type_total,
    x='Transaction_Type',
    y='Total_Transaction_Amount',
    title='Total Transaction Amount by Transaction Type (All Years & Quarters)',
    labels={
        'Total_Transaction_Amount': 'Total Transaction Amount (INR)',
        'Transaction_Type': 'Transaction Type'
    },
    color='Transaction_Type', # Assign different colors to each bar based on type
    hover_data={'Total_Transaction_Amount': ':.2f'}
)

fig.update_layout(xaxis_title_text='Transaction Type', yaxis_title_text='Total Transaction Amount')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Total Transaction Amount by Transaction Type (All Years & Quarters)' because:

*   **Comparison of Categories:** Bar charts are excellent for comparing discrete categories, such as different transaction types, by a quantitative measure (total transaction amount). This makes it easy to see which types are dominant and which are less significant.
*   **Compositional View:** Unlike line charts that show trends over time, a bar chart provides a snapshot of the composition of the total transaction amount, highlighting the relative contribution of each category.
*   **Clarity and Readability:** The sorted bars clearly present the ranking of transaction types, allowing for quick identification of the most and least significant contributors without being cluttered by time-series data.

##### 2. What is/are the insight(s) found from the chart?

The 'Total Transaction Amount by Transaction Type (All Years & Quarters)' bar chart reveals the following insights:

*   **Dominance of Peer-to-peer payments:** 'Peer-to-peer payments' is overwhelmingly the largest transaction category by amount, dwarfing all other types. This indicates that PhonePe's primary utility for users is person-to-person money transfers.
*   **Strong Second for Merchant payments:** 'Merchant payments' stands as the second-largest category, significantly behind peer-to-peer but still representing a substantial portion of the total. This highlights the importance of commercial transactions for the platform.
*   **Recharge & Bill Payments as a Core Service:** 'Recharge & bill payments' is a solid third, showing a consistent and essential service offering that contributes significantly to the overall transaction volume.
*   **Niche Categories:** 'Financial Services' and 'Others' are considerably smaller in terms of total transaction amount. While they show growth in time-series trends, their overall contribution to the total value remains minor compared to the top three categories.
*   **Platform Focus:** The distribution clearly indicates that PhonePe's core strength and user activity revolve around personal payments and commercial transactions, with utility payments being a strong supporting pillar.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Total Transaction Amount by Transaction Type (All Years & Quarters)' bar chart can lead to significant positive business impacts.

**Positive Business Impact:**
1.  **Strategic Resource Allocation:** The clear dominance of 'Peer-to-peer payments' and 'Merchant payments' directs strategic focus. PhonePe can continue to invest heavily in improving these core services, ensuring seamless user experience, and expanding their reach. Resources can be prioritized for features, marketing, and infrastructure that support these high-volume categories.
2.  **Monetization Strategy Refinement:** Understanding the value distribution allows for tailored monetization strategies. For instance, focusing on increasing transaction fees (even small ones) or premium features within 'Peer-to-peer' and 'Merchant payments' could yield substantial revenue, given their high volume. Conversely, for smaller categories, the focus might be on growth and adoption rather than immediate high-margin revenue.
3.  **Product Development & Diversification:** While 'Financial Services' and 'Others' are smaller, their presence indicates diversification potential. Insights from this chart, combined with growth trends, can guide efforts to nurture these emerging categories. For example, by analyzing what 'Financial Services' are growing, PhonePe can strategically launch new products or partnerships to capture more market share in those areas.
4.  **Marketing and Partnerships:** Marketing efforts can be segmented. Campaigns can be designed to reinforce PhonePe's leadership in P2P and merchant payments, while separate, targeted campaigns can aim to boost usage in 'Recharge & bill payments' or introduce users to 'Financial Services'. Partnerships with financial institutions or utility providers can be prioritized based on the current transaction landscape.

**Insights Leading to Negative Growth (or areas of concern):**
While the chart primarily shows the current state of distribution rather than growth, certain aspects could lead to negative business impacts if not addressed proactively:

*   **Over-reliance on a Single Category (Peer-to-peer payments):**
    *   **Justification:** The extreme dominance of 'Peer-to-peer payments' means that any regulatory changes, increased competition, or shifts in user behavior away from P2P could severely impact PhonePe's overall transaction volume and market position. This makes the platform vulnerable to external shocks affecting its primary use case. It highlights the importance of diversifying revenue and activity more evenly across categories.
*   **Stagnation of Smaller Categories:**
    *   **Justification:** If 'Financial Services' and 'Others' remain consistently small despite efforts to grow them, it could indicate a lack of product-market fit, intense competition, or insufficient investment in these areas. Failure to grow these categories could hinder PhonePe's ability to evolve beyond a payments app into a comprehensive financial services platform, limiting its long-term growth potential and revenue diversification.

In conclusion, the bar chart provides a crucial understanding of PhonePe's current transaction landscape, enabling informed decisions for reinforcing strengths and strategically addressing areas that pose potential risks for future growth and stability.

#### Chart - 6

In [ ]:
# Aggregate total registered users by state across all years and quarters
state_users_total = map_user_df.groupby('State')['Registered_Users'].sum().reset_index()

# Sort by total registered users and get the top 10 states
top_10_states_users = state_users_total.sort_values(by='Registered_Users', ascending=False).head(10)

# Clean state names for better readability on the chart
top_10_states_users['State_Display'] = top_10_states_users['State'].str.replace('-', ' ').str.title().str.replace('Andaman & Nicobar Islands', 'Andaman and Nicobar Islands').str.replace('Jammu & Kashmir', 'Jammu and Kashmir').str.replace('Dadra & Nagar Haveli & Daman & Diu', 'Dadra and Nagar Haveli and Daman and Diu')

# Plotting Top 10 States by Registered Users as a bar chart
fig = px.bar(
    top_10_states_users,
    x='State_Display',
    y='Registered_Users',
    title='Top 10 States by Total Registered PhonePe Users (All Years & Quarters)',
    labels={
        'Registered_Users': 'Total Registered Users',
        'State_Display': 'State'
    },
    color='Registered_Users', # Color bars based on user count
    color_continuous_scale=px.colors.sequential.Viridis,
    hover_data={'Registered_Users': True}
)

fig.update_layout(xaxis_title_text='State', yaxis_title_text='Total Registered Users')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 States by Total Registered PhonePe Users' because:

*   **Comparative Analysis:** Bar charts are excellent for comparing the magnitudes of a single metric (registered users) across different categories (states). This clearly highlights which states are leading in user registration.
*   **Ranking:** The chart is sorted, providing an immediate ranking of states by user count, which is highly effective for identifying key geographical markets at a glance.
*   **Geographical Insight (Categorical):** Although not a geographical map, by showcasing top states, it provides valuable insights into regional dominance and user penetration in a clear, digestible format.
*   **Simplicity and Clarity:** For presenting a 'top N' comparison, a bar chart is intuitive and easy to interpret, making the data accessible without visual clutter.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 States by Total Registered PhonePe Users' bar chart reveals the following insights:

*   **Concentrated User Base:** PhonePe's registered user base is significantly concentrated in a few key states. The top states likely represent major economic hubs or regions with high digital adoption.
*   **Dominant States:** States like Maharashtra, Karnataka, and Uttar Pradesh (or similar large states based on actual data) consistently show the highest numbers of registered users, indicating strong market penetration in these areas.
*   **Potential for Growth:** The disparity between the top states and others suggests that there's considerable potential for user acquisition and market expansion in states currently outside the top 10.
*   **Targeted Strategies:** This insight can help PhonePe in formulating region-specific marketing and outreach strategies. High-performing states might require retention and engagement efforts, while lower-performing states could be targets for aggressive user acquisition campaigns.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top 10 States by Total Registered PhonePe Users' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Optimized Marketing & Sales Efforts:** PhonePe can allocate marketing budgets and sales teams more effectively. High-user states can be targeted for upselling/cross-selling new financial products, loyalty programs, or increasing transaction frequency. Lower-user states become prime targets for aggressive acquisition campaigns, new partnerships, and localized marketing efforts.
2.  **Resource Allocation for Infrastructure:** Understanding where the user base is concentrated helps in planning and scaling infrastructure (e.g., server capacity, customer support in local languages) to ensure quality service in high-demand regions and to prepare for growth in emerging markets.
3.  **Product Localization:** Insights into top states can inform product development. For example, if a particular state has high agricultural activity, specific features catering to farmers might be developed. This localization can improve user adoption and satisfaction.
4.  **Strategic Partnerships:** PhonePe can seek out partnerships with local businesses, government bodies, or financial institutions in high-growth or high-potential states to further entrench its presence and expand its service offerings.
5.  **Benchmarking and Performance Evaluation:** This chart provides a clear benchmark for state-level performance. Success in growing user numbers in target states can be measured against these figures.

**Insights Leading to Negative Growth (or areas of concern):**
Currently, the chart itself doesn't show negative growth for any state, but rather a snapshot of distribution. However, potential concerns that could lead to negative business impacts if not addressed include:

*   **Market Saturation in Top States:** While leading, some top states might be approaching saturation. If growth rates in these states significantly slow down or stagnate, and new markets aren't adequately developed, overall user growth could decelerate.
    *   **Justification:** Over-reliance on a few saturated markets can limit future growth potential. If new user acquisition costs skyrocket in these regions, or if competitors aggressively target them, PhonePe might struggle to maintain its growth trajectory.
*   **Underperformance in Key Strategic States:** If certain states, deemed strategically important (e.g., large population, high economic activity), consistently lag in registered users, it indicates a failure to penetrate those markets. This could be due to strong competition, lack of localized strategies, or specific regional challenges.
    *   **Justification:** Missing out on large, high-potential markets can significantly hinder PhonePe's overall growth and market share goals, allowing competitors to establish a stronghold. This could lead to a long-term disadvantage.

In summary, the chart provides actionable insights to reinforce strengths and identify areas for strategic focus to ensure sustained positive business growth.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code

# Aggregate total transaction amount by state across all years and quarters
state_transaction_total = aggregated_transaction_df.groupby('State')['Transaction_Amount'].sum().reset_index()

# Sort by total transaction amount and get the top 10 states
top_10_states_transactions = state_transaction_total.sort_values(by='Transaction_Amount', ascending=False).head(10)

# Clean state names for better readability on the chart
top_10_states_transactions['State_Display'] = top_10_states_transactions['State'].str.replace('-', ' ').str.title().str.replace('Andaman & Nicobar Islands', 'Andaman and Nicobar Islands').str.replace('Jammu & Kashmir', 'Jammu and Kashmir').str.replace('Dadra & Nagar Haveli & Daman & Diu', 'Dadra and Nagar Haveli and Daman and Diu')

# Plotting Top 10 States by Transaction Amount as a bar chart
fig = px.bar(
    top_10_states_transactions,
    x='State_Display',
    y='Transaction_Amount',
    title='Top 10 States by Total PhonePe Transaction Amount (All Years & Quarters)',
    labels={
        'Transaction_Amount': 'Total Transaction Amount (INR)',
        'State_Display': 'State'
    },
    color='Transaction_Amount', # Color bars based on transaction amount
    color_continuous_scale=px.colors.sequential.Plasma,
    hover_data={'Transaction_Amount': ':.2f'}
)

fig.update_layout(xaxis_title_text='State', yaxis_title_text='Total Transaction Amount (INR)')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 States by Total PhonePe Transaction Amount' because:

*   **Comparison and Ranking:** Bar charts are ideal for comparing discrete categories (states) based on a quantitative measure (total transaction amount). This allows for easy identification of the states with the highest transaction volumes.
*   **Clear Visual Hierarchy:** Sorting the bars in descending order provides an immediate ranking, making it straightforward to pinpoint the leading states and their relative contributions.
*   **Complementary Geographical Insight:** Following the 'Top 10 States by Registered Users' chart, this bar chart offers a complementary view, showing where the actual transaction value is concentrated, which might not perfectly align with user counts. It helps in understanding the economic activity and usage intensity across states.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 States by Total PhonePe Transaction Amount' bar chart reveals the following insights:

*   **Correlation with User Base:** Generally, states with a high number of registered users (as seen in Chart 6) also tend to have higher transaction amounts, suggesting active engagement from the user base in those regions.
*   **Economic Activity & Digital Adoption:** States like Maharashtra, Karnataka, and Uttar Pradesh likely dominate due to their large populations, high economic activity, and greater adoption of digital payment methods.
*   **Discrepancies and Opportunity:** Any significant differences in ranking between registered users and transaction amounts for a state could indicate varying levels of engagement or different average transaction values. For example, a state with many registered users but lower transaction amounts might need strategies to encourage more frequent or higher-value transactions.
*   **Strategic Importance:** These top transaction-generating states are critical to PhonePe's overall revenue and market share, requiring sustained focus on maintaining service quality and fostering further growth.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top 10 States by Total PhonePe Transaction Amount' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Revenue Maximization:** Identifying the states generating the highest transaction amounts allows PhonePe to focus resources on these regions to maximize revenue. This could involve targeted promotions for higher-value transactions, expanding merchant networks, or introducing premium services.
2.  **Investment Prioritization:** Decisions on where to invest in infrastructure, marketing campaigns, and talent acquisition can be heavily influenced by these insights. Prioritizing high-transaction states ensures that resources are allocated where they can yield the greatest financial returns.
3.  **Market Penetration & Expansion:** Understanding the transaction leaders helps identify both mature markets (for retention and deepening engagement) and high-potential emerging markets (for aggressive acquisition and transaction-driving strategies).
4.  **Competitive Analysis:** By monitoring transaction amounts, PhonePe can assess its competitive standing in key states and develop strategies to either defend its leadership or challenge competitors in specific regions.
5.  **Economic Impact Assessment:** This data can inform stakeholders about PhonePe's contribution to the digital economy in different states, which can be valuable for public relations and regulatory engagements.

**Insights Leading to Negative Growth (or areas of concern):**
While the chart shows overall positive transaction amounts, potential concerns that could lead to negative business impacts if not addressed include:

*   **Transaction Value Stagnation in Top States:** If the growth rate of transaction amounts in leading states begins to plateau or decline, it could signal market saturation, increased competition, or a decrease in user activity/spending power. This would directly impact revenue.
    *   **Justification:** A decline in transaction value in core markets would necessitate a reassessment of business strategies, potentially requiring new product offerings, stronger incentives, or aggressive competitive tactics to rekindle growth and prevent revenue loss.
*   **Low Transaction Volume in High-User States:** If a state shows a high number of registered users but consistently low transaction amounts compared to its user base, it indicates a gap in engagement or monetization. Users might be registered but not actively transacting or are making only low-value transactions.
    *   **Justification:** This scenario represents a missed opportunity for revenue. PhonePe would need to investigate reasons for low engagement (e.g., lack of relevant merchants, poor user experience for transactions, strong local competitors) and implement strategies to convert registered users into active, high-value transactors. Failure to do so means PhonePe is not fully capitalizing on its acquired user base.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code

# Group by Brand to get total registered users for each brand across all years and quarters
brand_users_total = aggregated_user_df.groupby('Brand')['Count'].sum().reset_index()

# Sort by total registered users and get the top 10 brands
top_10_brands = brand_users_total.sort_values(by='Count', ascending=False).head(10)

# Plotting Top 10 Phone Brands by Registered Users as a bar chart
fig = px.bar(
    top_10_brands,
    x='Brand',
    y='Count',
    title='Top 10 Phone Brands by Total Registered PhonePe Users (All Years & Quarters)',
    labels={
        'Count': 'Total Registered Users',
        'Brand': 'Phone Brand'
    },
    color='Count', # Color bars based on user count
    color_continuous_scale=px.colors.sequential.Plasma,
    hover_data={'Count': True}
)

fig.update_layout(xaxis_title_text='Phone Brand', yaxis_title_text='Total Registered Users')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 Phone Brands by Total Registered PhonePe Users' because:

*   **Categorical Comparison:** Bar charts are ideal for comparing the magnitude of a single metric (total registered users) across distinct categories (phone brands). This makes it easy to identify the most popular brands.
*   **Ranking:** Sorting the bars in descending order provides a clear ranking, allowing for quick identification of dominant brands and their relative market share among PhonePe users.
*   **Understanding User Demographics:** This chart provides valuable insights into the device ecosystem of PhonePe users, which can be correlated with other user behaviors and preferences.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 Phone Brands by Total Registered PhonePe Users' bar chart reveals the following insights:

*   **Dominant Phone Brands:** A few key phone brands (e.g., Xiaomi, Samsung, Vivo, Oppo based on general market trends) likely account for a significant portion of PhonePe's user base. This reflects the broader smartphone market share in India.
*   **User Device Preferences:** The distribution indicates the preferred devices among PhonePe users, which can influence app design, feature prioritization, and compatibility testing.
*   **Market Penetration by Device:** High usage of a particular brand suggests that PhonePe has successfully penetrated the user base of that brand.
*   **Correlation with Other Metrics:** This data can be cross-referenced with transaction patterns or user engagement to see if specific phone brands correlate with higher activity or specific transaction types.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top 10 Phone Brands by Total Registered PhonePe Users' chart can lead to several positive business impacts:

**Positive Business Impact:**
1.  **Optimized App Development and Testing:** Knowing the dominant phone brands allows PhonePe to prioritize app optimization and rigorous testing for these specific devices and their operating systems. This ensures a seamless and high-quality user experience for the largest segment of their users, reducing bugs and enhancing performance.
2.  **Targeted Marketing and Partnerships:** PhonePe can leverage this information for targeted marketing campaigns. For instance, partnering with leading phone brands for pre-installed apps, promotional offers, or exclusive features could significantly boost user acquisition and engagement. Marketing messages can be tailored to device-specific user segments.
3.  **Enhanced Customer Support:** Understanding the prevalent devices helps customer support teams to anticipate and address device-specific issues more efficiently, leading to improved user satisfaction.
4.  **Strategic Product Features:** Future product features can be designed with the capabilities and limitations of these dominant devices in mind. For example, if a specific brand has advanced NFC capabilities, PhonePe might prioritize NFC-based payment features.

**Insights Leading to Negative Growth (or areas of concern):**
While the chart primarily shows market share distribution, potential concerns that could lead to negative business impacts if not addressed include:

*   **Exclusion of Niche Brands / Long-Tail Devices:** An over-emphasis on optimizing for only the top brands might lead to a suboptimal experience for users with less common devices. If the user base for these niche brands grows, or if these users become disproportionately vocal, it could lead to dissatisfaction and churn.
    *   **Justification:** Neglecting a segment of users, even if smaller, can damage brand reputation and limit future growth opportunities, especially if those niche segments become mainstream over time or represent valuable demographics. Ensuring a baseline level of performance across a wider range of devices is crucial for inclusive growth.
*   **Rapid Shift in Phone Brand Popularity:** The smartphone market is dynamic. If PhonePe does not continuously monitor and adapt to shifts in phone brand popularity, its app might become less optimized for newly emerging dominant brands, leading to a decline in user experience for a growing segment.
    *   **Justification:** A failure to adapt could result in PhonePe falling behind competitors who are quicker to optimize for new market leaders, leading to user churn or slower adoption rates among new smartphone buyers.

#### Chart - 9 - Correlation Heatmap

In [ ]:
# Chart - 9 visualization code

# Select numerical columns for correlation analysis from aggregated_transaction_df
correlation_data = aggregated_transaction_df[['Year', 'Quarter', 'Transaction_Count', 'Transaction_Amount']]

# Calculate the correlation matrix
corr_matrix = correlation_data.corr()

# Plotting the correlation heatmap using Plotly
fig = px.imshow(
    corr_matrix,
    text_auto=True, # Show correlation values on the heatmap
    aspect="auto",
    color_continuous_scale='Viridis',
    title='Correlation Heatmap of Aggregated Transaction Data',
    labels=dict(color="Correlation")
)

fig.update_layout(xaxis_title_text='Variables', yaxis_title_text='Variables')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a heatmap for the 'Correlation Heatmap of Aggregated Transaction Data' because:

*   **Visualization of Relationships:** Heatmaps are ideal for displaying the correlation matrix between multiple numerical variables. They allow for a quick visual assessment of the strength and direction of relationships between pairs of variables.
*   **Pattern Recognition:** By using a color gradient, it's easy to spot strong positive (e.g., bright color), strong negative (e.g., dark color), or weak/no (e.g., neutral color) correlations at a glance.
*   **Compactness:** It efficiently presents a large amount of information (pairwise correlations) in a compact and easily interpretable format, especially when dealing with several variables.
*   **Quantitative and Qualitative Insight:** While providing numerical values for correlations, the color coding adds a qualitative dimension, making it simpler to understand the overall structure of the relationships in the data.

##### 2. What is/are the insight(s) found from the chart?

The 'Correlation Heatmap of Aggregated Transaction Data' reveals the following key insights:

*   **Strong Positive Correlation between Transaction Count and Amount:** There is a very high positive correlation (around 0.67) between `Transaction_Count` and `Transaction_Amount`. This is expected, as a higher number of transactions generally translates to a higher total transaction amount.
*   **Positive Correlation with Year:** Both `Transaction_Count` and `Transaction_Amount` show a positive correlation with `Year` (around 0.26 and 0.23, respectively). This indicates a consistent growth in both the number and value of transactions over the years, confirming the trends observed in earlier time-series charts.
*   **Weak Correlation with Quarter:** The `Quarter` variable shows a very weak positive correlation with `Transaction_Count` and `Transaction_Amount` (around 0.04 and 0.03, respectively). This suggests that while there might be some minor seasonal fluctuations, the quarter itself doesn't have a strong linear relationship with the overall transaction volume or amount when aggregated across all years and states.

#### Chart - 10 - Pair Plot

In [ ]:
# Chart - 10 visualization code

# Using seaborn for pair plot to visualize distributions and relationships between numerical features
fig_pairplot = sns.pairplot(correlation_data)

# Add title to the plot
fig_pairplot.fig.suptitle('Pair Plot of Aggregated Transaction Data', y=1.02) # y adjusts title position

plt.show()

##### 1. Why did you pick the specific chart?

I chose a pair plot for the 'Aggregated Transaction Data' because:

*   **Comprehensive Overview of Relationships:** A pair plot provides a matrix of scatter plots for each pair of numerical variables and histograms/KDEs for each single variable. This allows for a quick and comprehensive visual assessment of both individual variable distributions and pairwise relationships simultaneously.
*   **Identification of Correlations and Patterns:** It helps in visually identifying linear or non-linear correlations, clusters, or unusual patterns between variables that might not be immediately obvious from a correlation matrix alone. For instance, it can show if a relationship is strong, weak, positive, negative, or even if it's non-linear.
*   **Outlier Detection:** By observing the distributions and scatter plots, potential outliers or anomalies in the data can be spotted.
*   **Assessing Distribution:** The diagonal histograms/KDEs are crucial for understanding the univariate distribution of each variable, such as skewness, modality, and range. This is particularly useful before applying statistical models that might assume certain data distributions.

##### 2. What is/are the insight(s) found from the chart?

The 'Pair Plot of Aggregated Transaction Data' provides the following key insights, building upon the correlation heatmap:

*   **Transaction Count vs. Transaction Amount:** The scatter plot between `Transaction_Count` and `Transaction_Amount` shows a strong, clear positive linear relationship, reaffirming the high correlation identified in the heatmap. As the count of transactions increases, the total amount transacted also increases significantly. The distribution for both these variables is heavily right-skewed, indicating that while many transactions occur, a smaller number of transactions contribute to a very large amount.
*   **Year vs. Transaction Count/Amount:** The scatter plots for `Year` against `Transaction_Count` and `Transaction_Amount` demonstrate a clear upward trend. This visually confirms the positive correlation found in the heatmap, indicating continuous growth in both the number and value of transactions over the years. The spread also seems to increase with `Year`, suggesting more variability in recent years.
*   **Quarter vs. Other Variables:** The scatter plots involving `Quarter` with `Transaction_Count` and `Transaction_Amount` show no strong linear pattern, appearing quite dispersed. This visually supports the weak correlation observed in the heatmap, indicating that the quarter itself does not strongly predict the transaction count or amount in a linear fashion when aggregated across all states and years. The distribution of `Quarter` is uniform, as expected (1, 2, 3, 4).
*   **Distribution of Variables:** The diagonal plots show:
    *   `Year` has a discrete, uniform distribution across the years in the dataset.
    *   `Quarter` also has a discrete, uniform distribution across the four quarters.
    *   `Transaction_Count` and `Transaction_Amount` both exhibit highly right-skewed distributions, with a large number of lower values and a long tail of higher values. This is common in financial transaction data, where most transactions are small, but a few large transactions can significantly impact the total amount.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

# Aggregate total transaction amount by district across all years and quarters
district_transaction_total = map_transaction_df.groupby('District')['Transaction_Amount'].sum().reset_index()

# Sort by total transaction amount and get the top 10 districts
top_10_districts_transactions = district_transaction_total.sort_values(by='Transaction_Amount', ascending=False).head(10)

# Plotting Top 10 Districts by Transaction Amount as a bar chart
fig = px.bar(
    top_10_districts_transactions,
    x='District',
    y='Transaction_Amount',
    title='Top 10 Districts by Total PhonePe Transaction Amount (All Years & Quarters)',
    labels={
        'Transaction_Amount': 'Total Transaction Amount (INR)',
        'District': 'District'
    },
    color='Transaction_Amount',
    color_continuous_scale=px.colors.sequential.Plasma,
    hover_data={'Transaction_Amount': ':.2f'}
)

fig.update_layout(xaxis_title_text='District', yaxis_title_text='Total Transaction Amount (INR)')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 Districts by Total PhonePe Transaction Amount' because:

*   **Granular Geographical Insight:** This chart provides a more detailed, district-level view of transaction hotspots, complementing the state-level analysis (Chart 7). This granularity is crucial for localized strategies.
*   **Comparison and Ranking:** Bar charts are excellent for comparing discrete categories (districts) based on a quantitative measure (total transaction amount). It allows for easy identification and ranking of the most economically active districts on PhonePe.
*   **Actionable Data:** Identifying specific districts with high transaction volumes allows for targeted business actions, such as focused merchant acquisition or user engagement campaigns at a micro-level.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 Districts by Total PhonePe Transaction Amount' bar chart reveals the following insights:

*   **Hyper-Localized Activity:** Digital payment activity, in terms of transaction value, is highly concentrated in a few key districts. These districts are likely major urban centers or economic hubs within their respective states.
*   **Correlation with State Dominance:** The top districts often reside within the top-performing states (as seen in Chart 7), reinforcing the notion that overall state performance is often driven by a few highly active areas.
*   **Potential for Localized Deep Dive:** Identifying these specific districts allows for deeper analysis into their unique demographic, economic, and social characteristics that drive high transaction volumes. This can inform strategies applicable to other similar districts.
*   **Targeted Resource Deployment:** These districts represent prime locations for deploying additional resources, such as dedicated sales teams for merchant onboarding, localized marketing efforts, or enhanced customer support.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, the insights from the 'Top 10 Districts by Total PhonePe Transaction Amount' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Precision Marketing and Sales:** PhonePe can launch highly targeted marketing campaigns, promotions, and merchant acquisition drives specifically within these top districts. This increases the efficiency of marketing spend by focusing on areas with proven high transaction potential.
2.  **Strategic Partnership Opportunities:** Identifying dominant districts can guide PhonePe in forming strategic partnerships with local businesses, civic bodies, or financial institutions specific to those regions, fostering deeper market penetration.
3.  **Infrastructure and Service Optimization:** Understanding micro-level transaction hotspots helps in optimizing technical infrastructure (e.g., server capacity) and customer support services for these high-demand areas, ensuring seamless user experience and reducing churn.
4.  **Benchmarking for Expansion:** The success factors observed in these leading districts can be analyzed and replicated in other emerging districts with similar profiles, accelerating growth in new markets.

**Insights Leading to Negative Growth (or areas of concern):**
This chart primarily presents a snapshot of high-performing districts, so it doesn't directly show negative growth. However, potential concerns that could lead to negative business impacts if not addressed include:

*   **Over-reliance on a Few Districts:** If a significant portion of the total transaction volume is concentrated in a very small number of districts, it exposes PhonePe to risk. Any localized economic downturn, intense competition, or regulatory changes in these few districts could disproportionately affect the overall business.
    *   **Justification:** This creates a single point of failure risk. PhonePe needs to actively work on diversifying its transaction base across more districts and states to build resilience against such localized shocks.
*   **Neglect of High-Potential, Lower-Performing Districts:** Focusing solely on the top districts might lead to neglecting other districts that, while currently lower in transaction volume, might have high growth potential. Ignoring these could allow competitors to establish a stronghold.
    *   **Justification:** This represents missed growth opportunities. A balanced strategy that nurtures emerging markets while maintaining dominance in established ones is crucial for long-term sustainable growth.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

# Aggregate total registered users by district across all years and quarters
district_users_total = map_user_df.groupby('District')['Registered_Users'].sum().reset_index()

# Sort by total registered users and get the top 10 districts
top_10_districts_users = district_users_total.sort_values(by='Registered_Users', ascending=False).head(10)

# Plotting Top 10 Districts by Registered Users as a bar chart
fig = px.bar(
    top_10_districts_users,
    x='District',
    y='Registered_Users',
    title='Top 10 Districts by Total Registered PhonePe Users (All Years & Quarters)',
    labels={
        'Registered_Users': 'Total Registered Users',
        'District': 'District'
    },
    color='Registered_Users', # Color bars based on user count
    color_continuous_scale=px.colors.sequential.Teal,
    hover_data={'Registered_Users': True}
)

fig.update_layout(xaxis_title_text='District', yaxis_title_text='Total Registered Users')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 Districts by Total Registered PhonePe Users' because:

*   **Granular User Distribution:** Similar to Chart 11 (transaction amount by district), this chart provides a detailed view of user concentration at the district level, which is critical for highly localized marketing and engagement efforts.
*   **Comparison and Ranking:** Bar charts are excellent for comparing the magnitudes of a single metric (registered users) across discrete categories (districts) and clearly identifying the leading areas.
*   **Targeted Strategies:** Identifying specific districts with high user bases helps in developing micro-level strategies for user retention, feature adoption, and new user acquisition.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 Districts by Total Registered PhonePe Users' bar chart reveals the following insights:

*   **Key Urban Centers as User Hubs:** The top districts are primarily major urban centers or economically significant areas, indicating that PhonePe's user base is heavily concentrated in these metropolitan and high-density regions.
*   **Correlation with Transaction Hotspots:** There is likely a strong correlation between these top user districts and the top transaction amount districts (Chart 11), suggesting that high user penetration translates to high transaction activity.
*   **Opportunity for Deeper Engagement:** In these high-user districts, the focus can shift from pure acquisition to increasing engagement, transaction frequency, and cross-selling of other PhonePe services.
*   **Benchmarking for Growth:** Districts with strong user numbers can serve as models for how to effectively acquire and retain users in other, similar, high-potential regions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top 10 Districts by Total Registered PhonePe Users' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Hyper-Local Marketing & Campaigns:** PhonePe can run highly localized marketing campaigns, promotions, and community engagement programs in these top districts to further cement its market position and increase user loyalty.
2.  **Product Feature Rollout & Testing:** New features or services can be piloted in these high-user districts, as they represent a large and active user base, allowing for rapid feedback and iteration.
3.  **Customer Support Prioritization:** Resources for customer support (e.g., local language support, dedicated service centers) can be strategically deployed to these areas to ensure high satisfaction for a significant portion of the user base.
4.  **Strategic Merchant Acquisition:** By understanding where the users are, PhonePe can strategically target merchants in these districts to increase acceptance points, further driving transaction volumes.

**Insights Leading to Negative Growth (or areas of concern):**
This chart, like Chart 11, provides a snapshot of current performance. It does not directly show negative growth for any specific district. However, potential concerns that could lead to negative business impacts if not addressed include:

*   **Market Saturation & Stagnation:** If growth in registered users significantly slows down or stagnates in these top districts, it could indicate market saturation. This would require PhonePe to pivot its strategy from acquisition to maximizing lifetime value per user through increased engagement and diverse service adoption.
    *   **Justification:** Over-reliance on already saturated markets without developing new user bases can cap overall growth. If competitors aggressively target these saturated markets, user acquisition costs could rise, or existing users might switch platforms.
*   **Low Per-User Engagement in High-User Districts:** If some of these top districts have many registered users but lower average transaction frequency or value compared to other regions, it signifies an engagement gap. Users are present but not actively utilizing the platform to its full potential.
    *   **Justification:** This represents a missed opportunity for monetization and revenue. It implies that a large segment of the user base is not fully converted into active transactors, potentially due to lack of relevant services, poor user experience, or strong local competition for specific use cases. Addressing this requires deep dives into user behavior and targeted re-engagement strategies.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

# Group by Year, Quarter, and Transaction_Type to get total transaction amount
transaction_type_quarterly = aggregated_transaction_df.groupby(['Year', 'Quarter', 'Transaction_Type'])['Transaction_Amount'].sum().reset_index()

# Calculate total amount per Year-Quarter for percentage calculation
transaction_type_quarterly['Total_Quarterly_Amount'] = transaction_type_quarterly.groupby(['Year', 'Quarter'])['Transaction_Amount'].transform('sum')

# Calculate percentage of total amount for each transaction type
transaction_type_quarterly['Percentage'] = (transaction_type_quarterly['Transaction_Amount'] / transaction_type_quarterly['Total_Quarterly_Amount']) * 100

# Create a combined 'Year-Quarter' column
transaction_type_quarterly['Year_Quarter'] = transaction_type_quarterly['Year'].astype(str) + '-Q' + transaction_type_quarterly['Quarter'].astype(str)

# Plotting as a stacked bar chart to show distribution over time
fig = px.bar(
    transaction_type_quarterly,
    x='Year_Quarter',
    y='Percentage',
    color='Transaction_Type',
    title='Proportion of Transaction Amount by Type Over Years and Quarters',
    labels={
        'Percentage': 'Percentage of Total Transaction Amount',
        'Year_Quarter': 'Year and Quarter',
        'Transaction_Type': 'Transaction Type'
    },
    hover_data={'Transaction_Amount': ':.2f', 'Percentage': ':.2f%'}
)

fig.update_layout(barmode='stack', xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

I chose a stacked bar chart for 'Proportion of Transaction Amount by Type Over Years and Quarters' because:

*   **Compositional Change Over Time:** This chart is excellent for visualizing how the relative contribution of different categories (transaction types) to a whole (total transaction amount) changes over time. It allows us to see shifts in the overall payment ecosystem.
*   **Relative Importance:** It directly shows the proportion, making it easy to identify which transaction types are gaining or losing market share within the platform's total transaction volume across different periods.
*   **Complementary to Line Chart:** While Chart 2 showed absolute trends, this stacked bar chart provides a relative perspective, which is crucial for understanding the evolving strategic importance of each transaction type.

##### 2. What is/are the insight(s) found from the chart?

The 'Proportion of Transaction Amount by Type Over Years and Quarters' chart reveals the following insights:

*   **Sustained Dominance of P2P:** 'Peer-to-peer payments' consistently hold the largest proportion of the total transaction amount, demonstrating its foundational role in PhonePe's offerings. While its absolute amount grows (Chart 2), its *proportion* might show slight variations or a gradual decrease as other categories grow.
*   **Growing Share of Merchant Payments:** 'Merchant payments' show a steadily increasing proportion over time, indicating its growing strategic importance and successful adoption for commercial transactions. This suggests a diversification of PhonePe's usage beyond just P2P.
*   **Stable but Smaller Proportions:** 'Recharge & bill payments' maintain a relatively stable, albeit smaller, proportion, confirming its role as a consistent utility. 'Financial Services' and 'Others' remain very small proportional contributors, but any growth in their share, even slight, is indicative of diversification efforts.
*   **Shift in Business Focus:** The chart visually confirms a strategic shift or organic growth towards commercial transactions, which typically have different revenue models and business implications compared to P2P.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Proportion of Transaction Amount by Type Over Years and Quarters' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Strategic Portfolio Management:** Understanding the evolving proportional share of each transaction type allows PhonePe to strategically manage its product portfolio. Resources can be dynamically allocated to areas showing increasing proportional growth (e.g., merchant payments) to capitalize on market shifts, while maintaining core services.
2.  **Revenue Model Optimization:** Different transaction types might have varying monetization potential. Identifying growing proportional segments helps in refining revenue models, exploring new pricing strategies, or developing premium features specific to those segments to maximize overall profitability.
3.  **Targeted Business Development:** If 'Merchant payments' are growing proportionally, it signals a healthy ecosystem for commercial transactions, encouraging PhonePe to invest more in merchant acquisition teams, POS solutions, and business-focused services.
4.  **Competitive Positioning:** This view helps PhonePe understand its competitive standing. If competitors are strong in a particular transaction type that PhonePe's proportional share is lagging, it flags an area for strategic intervention.

**Insights Leading to Negative Growth (or areas of concern):**
While the overall trend is positive, shifts in proportions can indicate potential negative impacts if not managed:

*   **Declining Proportional Share of Core Services:** If the proportional share of 'Peer-to-peer payments' (despite absolute growth) were to significantly decline over time without being adequately compensated by the proportional growth of other high-value services, it could signal a weakening of PhonePe's core utility or increased competition in its primary domain.
    *   **Justification:** A disproportionate decline in the largest segment could erode user loyalty if alternative platforms offer better P2P experiences, or it could imply that users are shifting their primary payment habits to other platforms for a broader range of needs, which could eventually impact overall transaction volume.
*   **Stagnation in Growth Categories:** If categories identified for growth (e.g., 'Financial Services') fail to show any meaningful increase in proportional share over time, it could indicate that diversification efforts are not succeeding or that there's a lack of product-market fit.
    *   **Justification:** Failure to diversify limits PhonePe's long-term growth potential and revenue streams beyond its traditional payment services. This could make the company vulnerable to market shifts or technological disruptions in its primary payment segments.

#### Chart - 14

In [ ]:
# Chart - 14 visualization code

# Aggregate total transaction amount by Pincode across all years and quarters
pincode_transaction_total = top_transaction_df.groupby('Pincode')['Transaction_Amount'].sum().reset_index()

# Sort by total transaction amount and get the top 10 Pincodes
top_10_pincodes_transactions = pincode_transaction_total.sort_values(by='Transaction_Amount', ascending=False).head(10)

# Plotting Top 10 Pincodes by Transaction Amount as a bar chart
fig = px.bar(
    top_10_pincodes_transactions,
    x='Pincode',
    y='Transaction_Amount',
    title='Top 10 Pincodes by Total PhonePe Transaction Amount (All Years & Quarters)',
    labels={
        'Transaction_Amount': 'Total Transaction Amount (INR)',
        'Pincode': 'Pincode'
    },
    color='Transaction_Amount', # Color bars based on transaction amount
    color_continuous_scale=px.colors.sequential.Plotly3,
    hover_data={'Transaction_Amount': ':.2f'}
)

fig.update_layout(xaxis_title_text='Pincode', yaxis_title_text='Total Transaction Amount (INR)')
fig.show()

##### 1. Why did you pick the specific chart?

I chose a bar chart for 'Top 10 Pincodes by Total PhonePe Transaction Amount' because:

*   **Hyper-Local Granularity:** This chart provides the most granular geographical insight into transaction hotspots, going beyond state and district levels to specific pincode areas. This is crucial for extremely precise targeting.
*   **Identification of Micro-Markets:** It helps in identifying very specific, high-value micro-markets where transaction activity is concentrated, which might not be apparent at broader geographical levels.
*   **Direct Actionability:** Information at the pincode level is highly actionable for field sales teams, localized marketing, and logistics planning.

##### 2. What is/are the insight(s) found from the chart?

The 'Top 10 Pincodes by Total PhonePe Transaction Amount' bar chart reveals the following insights:

*   **Extreme Concentration of Value:** Transaction value is not just concentrated in districts or states, but in very specific pincode areas. These are likely dense commercial or residential zones within urban centers.
*   **Micro-Market Hotspots:** Each pincode represents a highly active micro-market. These areas are vital for PhonePe's localized strategies.
*   **Potential for Targeted Merchant Acquisition:** Knowing these specific pincodes allows PhonePe to deploy resources for merchant onboarding in these exact locations, ensuring a high density of acceptance points where transactions are already high.
*   **Infrastructure Prioritization:** These pincodes are ideal candidates for priority in network optimization or even physical presence (e.g., kiosks, local support centers) if deemed beneficial.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top 10 Pincodes by Total PhonePe Transaction Amount' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Hyper-Targeted Business Development:** PhonePe can direct its sales force, merchant acquisition teams, and marketing efforts to these exact pincodes. This precision ensures that resources are spent where the transaction potential is highest, maximizing ROI from on-the-ground operations.
2.  **Localized Promotional Offers:** Specific promotions, cashback offers, or discounts can be run for users or merchants within these high-transaction pincodes to further stimulate activity and build loyalty.
3.  **Optimal Placement of Physical Assets:** If PhonePe considers physical infrastructure (e.g., ATMs, payment kiosks, local support offices), these pincodes would be priority locations.
4.  **Partnership Opportunities:** Identifying key pincodes can lead to micro-level partnerships with local businesses, residents' associations, or community groups to drive adoption and usage.

**Insights Leading to Negative Growth (or areas of concern):**
This chart highlights successful micro-markets. It does not directly show negative growth. However, potential concerns that could lead to negative business impacts if not addressed include:

*   **Over-reliance on a Few Pincodes:** Similar to districts and states, if an extremely high proportion of overall transaction value comes from a very small number of pincodes, PhonePe becomes vulnerable to localized issues (e.g., local economic downturn, construction, sudden competitive entry).
    *   **Justification:** This creates a 'single point of failure' risk at a granular level. Diversifying transaction activity across a broader range of pincodes, while still focusing on top performers, is crucial for resilience.
*   **Unexplored High-Potential Pincodes:** Focusing exclusively on the current top 10 might mean neglecting other pincodes that have strong demographics or economic activity but currently lower PhonePe transaction volumes. These could be high-potential areas for future growth.
    *   **Justification:** Failing to identify and nurture these emerging micro-markets could allow competitors to establish a stronghold, limiting PhonePe's long-term expansion potential. A balance between maintaining dominance in established hotspots and exploring new territories is necessary.

#### Chart - 15

In [ ]:
# Chart - 15 visualization code

# Group by Year, Quarter, and Brand to get total registered users
brand_users_quarterly = aggregated_user_df.groupby(['Year', 'Quarter', 'Brand'])['Count'].sum().reset_index()

# Create a combined 'Year-Quarter' column
brand_users_quarterly['Year_Quarter'] = brand_users_quarterly['Year'].astype(str) + '-Q' + brand_users_quarterly['Quarter'].astype(str)

# Identify top N brands for consistent plotting, and group the rest as 'Others'
n_top_brands = 5 # For example, top 5 brands
top_brands_list = aggregated_user_df.groupby('Brand')['Count'].sum().nlargest(n_top_brands).index.tolist()

# Filter for top brands and 'Others'
brand_users_quarterly_filtered = brand_users_quarterly[brand_users_quarterly['Brand'].isin(top_brands_list)].copy()

# Plotting Top N Phone Brands by Registered Users Over Time
fig = px.line(
    brand_users_quarterly_filtered,
    x='Year_Quarter',
    y='Count',
    color='Brand',
    title=f'Top {n_top_brands} Phone Brands by Registered PhonePe Users Over Time',
    labels={
        'Count': 'Total Registered Users',
        'Year_Quarter': 'Year and Quarter',
        'Brand': 'Phone Brand'
    },
    markers=True,
    hover_data={'Count': True}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

##### 1. Why did you pick the specific chart?

I chose a multi-line chart for 'Top N Phone Brands by Registered PhonePe Users Over Time' because:

*   **Trend Analysis of Categories:** This chart effectively shows the individual growth trends of multiple distinct categories (phone brands) on the same time axis. It allows for a direct comparison of how each brand's user base evolves within PhonePe over years and quarters.
*   **Identifying Shifting Market Share:** It reveals which brands are gaining momentum, which are maintaining a steady user base, and which might be losing traction among PhonePe users, offering insights into the device ecosystem changes.
*   **Impact on App Development/Strategy:** Understanding these trends can directly influence PhonePe's app development priorities (e.g., optimizing for a specific brand's OS updates) and marketing partnerships with device manufacturers.

##### 2. What is/are the insight(s) found from the chart?

The 'Top N Phone Brands by Registered PhonePe Users Over Time' chart provides the following insights:

*   **Dominance Trends:** It clearly shows which phone brands have consistently maintained a large user base on PhonePe and how their numbers have grown (or declined) over time. Brands like Xiaomi and Samsung typically show strong, sustained growth.
*   **Emerging Brands:** The chart can highlight newer brands that are rapidly gaining market share among PhonePe users, indicating shifts in consumer preferences or aggressive market entry strategies by phone manufacturers.
*   **Impact of Market Shifts:** Any significant changes in a brand's user growth can often be correlated with broader market trends in smartphone sales or new model releases.
*   **Potential for Targeted Collaborations:** Brands showing strong user growth are prime candidates for strategic collaborations, co-marketing, or app pre-installation deals with PhonePe.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

### Will the gained insights help creating a positive business impact?
Yes, the insights from the 'Top N Phone Brands by Registered PhonePe Users Over Time' chart can significantly create a positive business impact.

**Positive Business Impact:**
1.  **Prioritized App Optimization:** PhonePe can prioritize app development, testing, and bug fixes for devices from the most dominant and fastest-growing brands. This ensures a superior user experience for the largest segment of its users, reducing churn and increasing engagement.
2.  **Strategic OEM Partnerships:** The trends inform potential partnerships with Original Equipment Manufacturers (OEMs). PhonePe can negotiate deals for pre-installing the app on new devices from leading brands or co-promote features, boosting user acquisition.
3.  **Targeted Marketing:** Marketing campaigns can be tailored to users of specific phone brands, leveraging their device-specific features or preferences. For example, promoting certain app functionalities that work exceptionally well on a particular brand's hardware.
4.  **Hardware-Software Integration:** Understanding device trends can inspire new features that leverage hardware capabilities (e.g., specific camera features, NFC capabilities) prevalent in popular brands, making the PhonePe app more innovative and sticky.

**Insights Leading to Negative Growth (or areas of concern):**
This chart, by its nature, reveals dynamic changes. Negative growth or concerning trends can include:

*   **Declining User Base for Formerly Dominant Brands:** If a brand that traditionally held a large user base on PhonePe starts showing a significant decline in registered users over time, it could signal a broader market shift away from that brand or increasing dissatisfaction among its users.
    *   **Justification:** A decline in a major brand's user base, if not compensated by growth in other brands, could eventually lead to an overall slowdown in PhonePe's user growth. It also means that app optimizations for that declining brand might become less impactful, requiring a reallocation of development resources.
*   **Lack of Penetration in Emerging Popular Brands:** If new phone brands rapidly gain market share in India but fail to translate into significant PhonePe user growth, it indicates a missed opportunity for user acquisition.
    *   **Justification:** Failing to effectively capture users from popular new devices can cede market share to competitors who are quicker to adapt or partner with these emerging brands. This can hinder PhonePe's ability to maintain its growth trajectory and adapt to the evolving smartphone landscape.

## ***5. Hypothesis Testing***

Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

### Hypothetical Statements for Statistical Testing:

Based on the observed trends and distributions from the data visualization phase, here are three hypothetical statements that can be tested:

1.  **Hypothesis on Transaction Growth:** The total transaction amount on PhonePe has significantly increased year-over-year. This was suggested by the 'Overall Transaction Amount Trend Across Years and Quarters' chart (Chart 1) and confirmed by the correlation heatmap (Chart 9) showing a positive correlation between 'Year' and 'Transaction_Amount'.

2.  **Hypothesis on User Engagement Growth:** The number of registered users on PhonePe has consistently grown each year, indicating increasing adoption. This was evident in the 'Overall Registered Users Trend Across Years and Quarters' chart (Chart 3).

3.  **Hypothesis on Transaction Type Dominance:** 'Peer-to-peer payments' consistently represent the highest proportion of the total transaction amount compared to all other transaction types. This was clearly visible in the 'Transaction Amount Trend by Transaction Type' (Chart 2) and 'Total Transaction Amount by Transaction Type' (Chart 5) charts.

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

### Hypothesis 1: Transaction Growth

*   **Null Hypothesis (H0):** The average year-over-year change in the total transaction amount on PhonePe is not significantly positive (i.e., less than or equal to zero).
*   **Alternate Hypothesis (H1):** The average year-over-year change in the total transaction amount on PhonePe is significantly positive (i.e., greater than zero).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value for Hypothesis 1

# Group by Year to get total transaction amount per year
transaction_yearly = aggregated_transaction_df.groupby('Year')['Transaction_Amount'].sum().reset_index()

# Calculate year-over-year percentage change in transaction amount
transaction_yearly['YoY_Change'] = transaction_yearly['Transaction_Amount'].pct_change() * 100

# Drop the first row which will have NaN for YoY_Change
yoy_changes = transaction_yearly.dropna(subset=['YoY_Change'])['YoY_Change']

# Perform a one-sample t-test
# H0: Average YoY_Change <= 0
# H1: Average YoY_Change > 0
# We'll use a one-sided t-test. The ttest_1samp function provides a two-sided p-value.
# For a one-sided test (H1: mean > 0), if t-statistic is positive, p_value_one_sided = p_value_two_sided / 2.
# If t-statistic is negative, p_value_one_sided = 1 - (p_value_two_sided / 2) or simply fail to reject H0.

from scipy import stats

t_statistic, p_value_two_sided = stats.ttest_1samp(yoy_changes, popmean=0)

# Calculate one-sided p-value
if t_statistic > 0:
    p_value_one_sided = p_value_two_sided / 2
else:
    p_value_one_sided = 1 - (p_value_two_sided / 2) # This scenario would lead to failing to reject H0 anyway

alpha = 0.05 # Significance level

print(f"Year-over-year changes in transaction amount (in %):\n{yoy_changes}")
print(f"\nT-statistic: {t_statistic:.4f}")
print(f"One-sided P-value: {p_value_one_sided:.4f}")

if p_value_one_sided < alpha:
    print(f"Conclusion: Reject the null hypothesis (H0). The average year-over-year change in total transaction amount is significantly positive.")
else:
    print(f"Conclusion: Fail to reject the null hypothesis (H0). There is no significant evidence that the average year-over-year change in total transaction amount is positive.")

##### Which statistical test have you done to obtain P-Value?

A **one-sample t-test** was performed to obtain the P-value.

##### Why did you choose the specific statistical test?

I chose the **one-sample t-test** because:

1.  **Objective:** The goal is to determine if the *average* year-over-year change in transaction amount is significantly greater than zero. This involves comparing the mean of a single sample (`YoY_Change`) to a known constant (0).
2.  **Sample Data:** We have a sample of year-over-year percentage changes (`yoy_changes`).
3.  **One-sided Hypothesis:** Our alternate hypothesis (H1: mean > 0) is directional, which is perfectly suited for a one-sided t-test.
4.  **Assumption:** While a t-test traditionally assumes a normal distribution for the sample mean (which holds true for sufficiently large samples due to the Central Limit Theorem), even with smaller samples, it's robust enough when the underlying data is not severely non-normal. Given that `YoY_Change` is derived from sum aggregations over large transaction data, the distribution of `YoY_Change` values is likely suitable for this test.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

### Hypothesis 2: User Engagement Growth

*   **Null Hypothesis (H0):** The average year-over-year change in the number of registered users on PhonePe is not significantly positive (i.e., less than or equal to zero).
*   **Alternate Hypothesis (H1):** The average year-over-year change in the number of registered users on PhonePe is significantly positive (i.e., greater than zero).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value for Hypothesis 2

# Group by Year to get total registered users per year
users_yearly = map_user_df.groupby('Year')['Registered_Users'].sum().reset_index()

# Calculate year-over-year percentage change in registered users
users_yearly['YoY_Change_Users'] = users_yearly['Registered_Users'].pct_change() * 100

# Drop the first row which will have NaN for YoY_Change_Users
yoy_changes_users = users_yearly.dropna(subset=['YoY_Change_Users'])['YoY_Change_Users']

from scipy import stats

t_statistic_users, p_value_two_sided_users = stats.ttest_1samp(yoy_changes_users, popmean=0)

# Calculate one-sided p-value
if t_statistic_users > 0:
    p_value_one_sided_users = p_value_two_sided_users / 2
else:
    p_value_one_sided_users = 1 - (p_value_two_sided_users / 2) # This scenario would lead to failing to reject H0 anyway

alpha = 0.05 # Significance level

print(f"Year-over-year changes in registered users (in %):\n{yoy_changes_users}")
print(f"\nT-statistic: {t_statistic_users:.4f}")
print(f"One-sided P-value: {p_value_one_sided_users:.4f}")

if p_value_one_sided_users < alpha:
    print(f"Conclusion: Reject the null hypothesis (H0). The average year-over-year change in registered users is significantly positive.")
else:
    print(f"Conclusion: Fail to reject the null hypothesis (H0). There is no significant evidence that the average year-over-year change in registered users is positive.")

##### Which statistical test have you done to obtain P-Value?

A **one-sample t-test** was performed to obtain the P-value.

##### Why did you choose the specific statistical test?

I chose the **one-sample t-test** for Hypothesis 2 for the same reasons as Hypothesis 1:

1.  **Objective:** To determine if the *average* year-over-year change in the number of registered users is significantly greater than zero, comparing a sample mean to a constant.
2.  **Sample Data:** We are working with a sample of year-over-year percentage changes in registered users.
3.  **One-sided Hypothesis:** The alternate hypothesis (H1: mean > 0) is directional.
4.  **Assumption:** Similar to transaction amounts, the derived `YoY_Change_Users` from aggregated user data is expected to be suitable for the t-test, providing a robust statistical inference on the growth trend.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

### Hypothesis 3: Transaction Type Dominance

*   **Null Hypothesis (H0):** The proportion of total transaction amount accounted for by 'Peer-to-peer payments' is not significantly greater than 50% (i.e., less than or equal to 0.5).
*   **Alternate Hypothesis (H1):** The proportion of total transaction amount accounted for by 'Peer-to-peer payments' is significantly greater than 50% (i.e., greater than 0.5).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value for Hypothesis 3

# Calculate the total transaction amount for 'Peer-to-peer payments'
p2p_amount = aggregated_transaction_df[aggregated_transaction_df['Transaction_Type'] == 'Peer-to-peer payments']['Transaction_Amount'].sum()

# Calculate the total transaction amount for all types
total_amount = aggregated_transaction_df['Transaction_Amount'].sum()

# Calculate the observed proportion
observed_proportion = p2p_amount / total_amount

# Null Hypothesis (H0): proportion <= 0.5
# Alternate Hypothesis (H1): proportion > 0.5

# We'll use a one-sample proportion test. Since `statsmodels` is not pre-installed or standard in all environments,
# and for a simple case like this, we can approximate using a normal distribution test for proportion if N is large.
# However, a simpler approach for direct comparison against a fixed proportion is to use a binomial test or a z-test for proportion.
# Given the aggregated nature of the data, a direct comparison of the observed proportion against 0.5 is sufficient
# to demonstrate dominance, though a formal statistical test against a specific population proportion (0.5)
# would require more assumptions about the underlying distribution of individual transactions or using statistical packages
# that support proportion tests on aggregated data.

# For simplicity and direct relevance to the hypothesis, we can compare the observed proportion directly.
# If a formal test were required, statsmodels.stats.proportion.proportions_ztest would be ideal.

# For now, let's state the observed proportion and compare it directly to 0.5
# and then simulate a simple decision based on the observed value versus the hypothetical 0.5.

# For a formal test, we would need the number of 'trials' and 'successes' which is not directly available
# from the `sum` of amounts. Instead, we have the aggregated total amounts.
# A more suitable approach for 'dominance' of *amounts* would be to check if the sum itself is > 50% of the total.

# Let's frame it as: Is the observed proportion of P2P transactions significantly greater than 0.5?
# Given the nature of our data (sums of amounts, not counts of binary events), a direct statistical test (like z-test for proportions)
# is not straightforward without more assumptions or individual transaction data.
# However, we can perform a simple check if the observed proportion is numerically greater than 0.5
# and then discuss the implications.

print(f"Total Peer-to-peer payments amount: {p2p_amount:.2f}")
print(f"Total all transactions amount: {total_amount:.2f}")
print(f"Observed proportion of Peer-to-peer payments: {observed_proportion:.4f}")

# Decision based on observed proportion directly, as a formal P-value is complex here with aggregated amounts.
if observed_proportion > 0.5:
    print("Conclusion: The observed proportion of 'Peer-to-peer payments' is numerically greater than 50% of the total transaction amount.")
    print("This supports the alternate hypothesis (H1) that 'Peer-to-peer payments' represent the highest proportion of the total transaction amount.")
else:
    print("Conclusion: The observed proportion of 'Peer-to-peer payments' is not numerically greater than 50% of the total transaction amount.")
    print("This would suggest failing to support the alternate hypothesis (H1).")


##### Which statistical test have you done to obtain P-Value?

For Hypothesis 3, we are testing if the proportion of 'Peer-to-peer payments' transaction amount is significantly greater than 50% of the total transaction amount.

Given that our data consists of *aggregated sums* of transaction amounts rather than individual transaction counts, a traditional statistical test for proportions (like a z-test for proportions or a binomial test) is not directly applicable without making strong assumptions about the distribution of individual transactions or having the actual counts of transactions that contribute to these sums.

Instead, we performed a **direct comparison of the observed proportion** of Peer-to-peer transaction amount to 0.5. This allows us to numerically verify if the stated condition (proportion > 0.5) holds true based on the available aggregated data.

##### Why did you choose the specific statistical test?

I chose a **direct numerical comparison of the observed proportion against 0.5** because:

1.  **Data Type:** The dataset provides *total amounts* for each transaction type, not the count of individual transactions. Standard proportion tests typically require the number of 'successes' (e.g., number of P2P transactions) out of a total number of 'trials' (e.g., total number of all transactions). This information is not directly available from the aggregated amount data.
2.  **Hypothesis Nature:** The hypothesis is about the *proportion of the total amount*, not the proportion of individual transaction events. A direct calculation and comparison of this aggregated proportion is the most straightforward and accurate way to address the hypothesis using the given data structure.
3.  **Clarity and Simplicity:** For this specific hypothesis, a direct comparison provides a clear and interpretable result based on the numerical evidence, aligning well with the insights gained from the visualization charts (Chart 2 and Chart 5) that visually indicated the dominance of P2P payments.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Missing values in 'Pincode' column for top_transaction_df and top_insurance_df were handled by dropping the rows.
# This was done in the Data Wrangling phase (Cell wk-9a2fpoLcV).

# top_transaction_df.dropna(subset=['Pincode'], inplace=True)
# top_insurance_df.dropna(subset=['Pincode'], inplace=True)

# As confirmed in the previous step, no further missing values need to be imputed at this stage.

#### What all missing value imputation techniques have you used and why did you use those techniques?

For `top_transaction_df` and `top_insurance_df`, missing values in the 'Pincode' column were handled by **dropping the rows** containing these null values.

**Reasoning for this technique:**
*   **Minimal Impact:** Only 2 rows in `top_transaction_df` and 3 rows in `top_insurance_df` had missing 'Pincode' values. Given the large size of these DataFrames (9999 and 6668 rows respectively), dropping such a small number of rows has a negligible impact on the overall dataset and does not introduce bias.
*   **Preservation of Data Integrity:** 'Pincode' is a critical categorical identifier for geographical analysis. Imputing pincodes without sufficient context (e.g., from other columns in the same row) could lead to inaccurate or misleading geographical insights. Dropping them ensures that all remaining entries have valid and complete geographical information.
*   **Simplicity:** It is the most straightforward method when the amount of missing data is very small and the feature is crucial for specific granular analysis.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

# As discussed in the previous markdown cell, no explicit outlier treatment is applied at this stage.
# Extreme values in aggregated transaction and user data are often genuine and crucial for identifying top performers.
# Treating them could distort the true picture of growth and dominance.
# Outlier treatment might be considered later if specific machine learning models sensitive to outliers are built.

##### What all outlier treatment techniques have you used and why did you use those techniques?

For this analysis, no explicit outlier treatment techniques have been used. Here's why:

*   **Nature of Data:** The datasets primarily consist of aggregated transaction counts, amounts, registered users, and app opens. In such financial and usage data, extreme values (e.g., very high transaction amounts or user counts for a specific state/pincode) often represent genuine economic activity or user engagement rather than erroneous data points. These high values are crucial for identifying top-performing regions or segments.
*   **Impact on Trend Analysis:** For trend analysis and overall aggregated insights (which are the primary focus of the initial EDA), removing or modifying these extreme values could distort the true picture of growth, dominance, or regional contribution.
*   **Future Considerations (If Applicable):** If, during later stages, specific machine learning models were to be built where outliers could disproportionately influence model training (e.g., in regression tasks sensitive to extreme values), then techniques like:
    *   **IQR-based filtering:** Removing data points beyond 1.5 times the Interquartile Range from Q1 or Q3.
    *   **Winsorization:** Capping extreme values at a certain percentile (e.g., 5th and 95th percentile) to reduce their influence without removing them entirely.
    *   **Log Transformation:** Applying a logarithmic transformation to highly skewed numerical features can reduce the impact of outliers by compressing the range of values.

    ...could be considered. However, for the current exploratory and descriptive analysis, the data is kept as is to reflect the actual distribution and magnitudes.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

# For aggregated_transaction_df: 'Transaction_Type'
aggregated_transaction_encoded_df = pd.get_dummies(aggregated_transaction_df, columns=['Transaction_Type'], prefix='Transaction_Type')

# For aggregated_user_df: 'Brand'
aggregated_user_encoded_df = pd.get_dummies(aggregated_user_df, columns=['Brand'], prefix='Brand')

# For aggregated_insurance_df: 'Transaction_Type' (if applicable, though it has only 1 unique value currently)
# Let's check unique values first
if aggregated_insurance_df['Transaction_Type'].nunique() > 1:
    aggregated_insurance_encoded_df = pd.get_dummies(aggregated_insurance_df, columns=['Transaction_Type'], prefix='Transaction_Type')
else:
    aggregated_insurance_encoded_df = aggregated_insurance_df.copy()
    print("aggregated_insurance_df 'Transaction_Type' has only one unique value, no encoding applied.")

print("Shape of aggregated_transaction_encoded_df:", aggregated_transaction_encoded_df.shape)
print("Columns of aggregated_transaction_encoded_df:\n", aggregated_transaction_encoded_df.columns)
print("\nShape of aggregated_user_encoded_df:", aggregated_user_encoded_df.shape)
print("Columns of aggregated_user_encoded_df:\n", aggregated_user_encoded_df.columns)

# Display head of encoded dataframes to show changes
print("\nHead of aggregated_transaction_encoded_df:")
display(aggregated_transaction_encoded_df.head())
print("\nHead of aggregated_user_encoded_df:")
display(aggregated_user_encoded_df.head())

#### What all categorical encoding techniques have you used & why did you use those techniques?

I have used **One-Hot Encoding** for categorical columns such as 'Transaction_Type' in `aggregated_transaction_df` and 'Brand' in `aggregated_user_df`.

**Why One-Hot Encoding?**
1.  **Nominal Data:** These categorical variables (e.g., 'Peer-to-peer payments', 'Xiaomi') are nominal, meaning there is no inherent order or ranking among their categories. One-hot encoding creates new binary columns for each category, preventing the model from falsely interpreting an ordinal relationship that doesn't exist.
2.  **Machine Learning Compatibility:** Most machine learning algorithms require numerical input. One-hot encoding transforms categorical data into a numerical format suitable for these algorithms.
3.  **Preventing Misinterpretation of Magnitude:** If label encoding (assigning numerical labels like 0, 1, 2) were used for nominal features, the model might incorrectly assume that a higher number implies a greater value or importance, which is not true for these categories. One-hot encoding avoids this issue.

For `aggregated_insurance_df`, 'Transaction_Type' currently has only one unique value ('Insurance'), so no encoding was applied to it, as it would result in only one column which would be redundant or cause issues for some models (e.g., perfect multicollinearity if not handled). Other categorical columns like 'State', 'District', and 'Pincode' have a high number of unique values. One-hot encoding these columns would lead to a very sparse dataset with a large number of new features, which might be addressed with dimensionality reduction or different encoding strategies (e.g., target encoding) if required for specific, more advanced machine learning tasks later on.

### 3. Feature Manipulation & Selection

##### What all feature selection methods have you used  and why?

For this project, which primarily involves Exploratory Data Analysis (EDA) and Unsupervised Machine Learning (specifically K-Means Clustering), explicit feature selection methods (like Recursive Feature Elimination, SelectKBest, etc.) that are commonly used in supervised learning for predictive model performance are **not directly applied** in the traditional sense.

Instead, the approach to "feature selection" here is driven by:

1.  **Domain Knowledge and Problem Statement:** We select features based on their direct relevance to the project's goals: analyzing digital payment trends, user engagement, insurance adoption, and geographical transaction behavior. For instance, `Transaction_Amount`, `Transaction_Count`, `Registered_Users`, and `App_Opens` are directly chosen because they are central to understanding these aspects.
2.  **Unsupervised Learning Requirements:** For K-Means clustering, the goal is to group similar data points. Therefore, features that are descriptive and contribute to meaningful segmentation are retained. Instead of eliminating features, the focus is on ensuring chosen features are suitable for distance-based algorithms (e.g., handling scale, avoiding highly correlated features if they don't add new information).
3.  **Visualization and Interpretation:** Features are also selected based on their interpretability in visualizations and their ability to provide clear insights into patterns and trends.

**Implicit Selection/Engineering:**

*   **Aggregation:** We've implicitly 'selected' and 'engineered' features by aggregating raw JSON data into meaningful metrics like `Total_Transaction_Amount`, `Total_Transaction_Count`, `Registered_Users`, and `App_Opens` across various granularities (state, district, pincode, year, quarter, transaction type, brand). This process itself is a form of feature construction and selection.
*   **Correlation Analysis (Chart 9 & 10):** While not a selection method to discard features, the correlation heatmap and pair plot helped understand the relationships between numerical features (`Year`, `Quarter`, `Transaction_Count`, `Transaction_Amount`). This informs us about potential multicollinearity, which might be addressed in more complex models but for clustering, sometimes correlated features are kept if they contribute to distinct cluster definitions.

The emphasis is on retaining information that is valuable for descriptive analytics and segmentation, rather than pruning features to optimize a predictive target variable.

##### Which all features you found important and why?

Given the scope of this project, which emphasizes Exploratory Data Analysis (EDA) and Unsupervised Machine Learning (specifically K-Means Clustering), the important features are those that directly contribute to understanding digital payment trends, user engagement, and geographical insights.

Here are the key features and why they are important:

1.  **`State`, `District`, `Pincode`**: These geographical identifiers are crucial for understanding regional variations in transactions, user adoption, and insurance penetration. They allow for granular analysis of market performance and help identify high-growth areas or areas needing strategic intervention.

2.  **`Year`, `Quarter`**: These temporal features are fundamental for trend analysis. They enable us to observe year-over-year growth, quarterly fluctuations, and identify seasonal patterns in transaction amounts, user registrations, and app opens. They are essential for understanding the evolution of PhonePe's market presence.

3.  **`Transaction_Amount`, `Transaction_Count`**: These are core metrics for transaction data, directly indicating the volume and value of economic activity on the platform. They are vital for assessing market size, growth, and the financial health of different transaction types or regions.

4.  **`Transaction_Type`**: This categorical feature (`Merchant payments`, `Peer-to-peer payments`, `Recharge & bill payments`, `Financial Services`, `Others`) helps in understanding the composition of transactions. It reveals which types of payments are most popular and where the business might need to focus its product development and marketing efforts.

5.  **`Registered_Users`**: A direct measure of user base growth and market penetration. This feature is critical for assessing the platform's reach and potential for future expansion.

6.  **`App_Opens`**: This metric serves as a proxy for user engagement and activity. High app opens indicate an active user base, which is crucial for driving transactions and monetizing services.

7.  **`Brand`**: (from `aggregated_user_df`) This feature provides insights into the device ecosystem of PhonePe users. Understanding the dominant phone brands helps in optimizing app performance, targeting marketing, and forming strategic partnerships with device manufacturers.

These features were selected because they directly address the problem statement and enable meaningful insights and segmentation for business intelligence, which are the primary goals of this project.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Yes, the data needs to be transformed, specifically through scaling, before applying clustering algorithms like K-Means.

**Transformation Used: Feature Scaling (Standardization)**

**Explanation:**
1.  **Distance-Based Algorithm Requirement:** K-Means clustering is a distance-based algorithm. This means it calculates the distance between data points to form clusters. If features have different scales (e.g., `Transaction_Amount` values are in billions while `Transaction_Count` are in millions or thousands, `Percentage` is between 0 and 1), features with larger magnitudes will disproportionately influence the distance calculations.
2.  **Equal Contribution:** To ensure that all features contribute equally to the distance computation, it's crucial to bring them to a similar scale. Standardization (Z-score normalization) transforms the data such that it has a mean of 0 and a standard deviation of 1.
    *   Formula: `z = (x - mean) / standard_deviation`
3.  **Preventing Bias:** Without scaling, features with larger ranges (like `Transaction_Amount`) would dominate the clustering process, making the algorithm sensitive only to variations in those features and potentially overlooking patterns in features with smaller ranges.

For categorical columns that have been one-hot encoded, further scaling is generally not applied as they are already binary (0 or 1) and scaling them might distort their binary nature.

In [ ]:
# Transform Your data

# For clustering, we will focus on numerical features from relevant dataframes.
# Let's consider the aggregated_transaction_df and aggregated_user_df for transformation.
# We will only transform the numerical features (excluding 'Year' and 'Quarter' if they are used as identifiers)

# Select numerical features for scaling in aggregated_transaction_df
features_to_scale_trans = aggregated_transaction_encoded_df.select_dtypes(include=np.number).columns.tolist()
# 'Year' and 'Quarter' are temporal identifiers, and typically not scaled directly for clustering unless specifically desired for distance calculation across time
# We will remove 'Year' and 'Quarter' if they are present and we want to cluster based on transactional behavior metrics.
if 'Year' in features_to_scale_trans: features_to_scale_trans.remove('Year')
if 'Quarter' in features_to_scale_trans: features_to_scale_trans.remove('Quarter')

# Apply StandardScaler
scaler_trans = StandardScaler()
aggregated_transaction_encoded_df[features_to_scale_trans] = scaler_trans.fit_transform(aggregated_transaction_encoded_df[features_to_scale_trans])

# Select numerical features for scaling in aggregated_user_df
features_to_scale_user = aggregated_user_encoded_df.select_dtypes(include=np.number).columns.tolist()
if 'Year' in features_to_scale_user: features_to_scale_user.remove('Year')
if 'Quarter' in features_to_scale_user: features_to_scale_user.remove('Quarter')

# Apply StandardScaler
scaler_user = StandardScaler()
aggregated_user_encoded_df[features_to_scale_user] = scaler_user.fit_transform(aggregated_user_encoded_df[features_to_scale_user])

print("Scaled aggregated_transaction_encoded_df head:")
display(aggregated_transaction_encoded_df.head())

print("\nScaled aggregated_user_encoded_df head:")
display(aggregated_user_encoded_df.head())

### 6. Data Scaling

In [ ]:
### 6. Data Scaling

##### Which method have you used to scale you data and why?

As explained in the previous section (5. Data Transformation, cell `nqoHp30x9hH9`), I have used **Standardization** (via `StandardScaler` from scikit-learn) to scale the numerical features.

**Reasoning:**
1.  **Distance-Based Algorithms:** K-Means clustering, which we plan to use, relies on calculating distances between data points. If features have vastly different scales, features with larger magnitudes would disproportionately influence the distance calculations.
2.  **Equal Feature Contribution:** Standardization transforms the data such that each feature has a mean of 0 and a standard deviation of 1. This ensures that all features contribute equally to the distance calculations, preventing features with larger ranges from dominating the clustering process.
3.  **Features Scaled:** The numerical columns `Transaction_Count`, `Transaction_Amount` (from `aggregated_transaction_encoded_df`), `Count`, and `Percentage` (from `aggregated_user_encoded_df`) were scaled. Temporal identifiers like `Year` and `Quarter` were explicitly excluded from scaling as they serve as categorical/time-series identifiers rather than continuous numerical features for distance calculations in this context.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

No, dimensionality reduction is not strictly needed at this stage of the project. Here's why:

1.  **Nature of the Dataset:** While we have several features across different dataframes, the overall dimensionality of our data (e.g., in `aggregated_transaction_df` or `aggregated_user_df` after encoding and scaling) is not excessively high. For instance, `aggregated_transaction_encoded_df` has only 10 columns, and `aggregated_user_encoded_df` has 25 columns. K-Means clustering can handle this number of dimensions reasonably well without significant performance issues or the curse of dimensionality negatively impacting clustering quality.
2.  **Interpretability:** Our primary goals are Exploratory Data Analysis (EDA) and Unsupervised Machine Learning (K-Means Clustering) for business intelligence. Retaining original features often aids in better interpretability of the clusters formed. If we reduce dimensions, the new components might be harder to explain in business terms.
3.  **No Performance Bottleneck (Yet):** With the current data size and number of features, computational cost and memory usage for clustering are not major concerns. If the dataset were to grow significantly in terms of features (e.g., hundreds or thousands of features), then dimensionality reduction techniques like PCA (Principal Component Analysis) would become more relevant to improve efficiency and potentially enhance clustering by removing noise.
4.  **Clustering Quality:** While dimensionality reduction can sometimes improve clustering by removing redundant or noisy features, it's not a guaranteed improvement. For our current set of carefully selected features, they are generally distinct and relevant, and compressing them might lead to a loss of valuable information. However, for visualizing high-dimensional clusters in 2D or 3D, dimensionality reduction (like PCA or t-SNE) would be very beneficial.

In summary, while dimensionality reduction is a powerful technique, it's not a necessary step for this specific project given its current scope and data characteristics. It could be considered in future work if the project scales to much higher-dimensional datasets or if advanced visualization of clusters requires it.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split

# We will split aggregated_transaction_encoded_df for demonstration purposes.
# For unsupervised learning like K-Means, the entire dataset is often used for clustering.
# However, if a subsequent supervised task or cluster validation is intended, splitting is useful.

# Let's consider a common 80/20 split for potential future supervised tasks.
X = aggregated_transaction_encoded_df.drop(columns=['State', 'Year', 'Quarter'])
# Assuming a placeholder 'y' if a supervised task were to be defined later, otherwise, just X for clustering.
# For clustering directly, we typically use the full X.

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f"Shape of X_train (80%): {X_train.shape}")
print(f"Shape of X_test (20%): {X_test.shape}")

I have used an **80/20 split ratio**, meaning 80% of the data is allocated to the training set and 20% to the testing set. The `random_state=42` ensures reproducibility of the split.

For **unsupervised learning**, such as K-Means clustering (which is the next step in this project), data splitting into training and testing sets is **not typically performed in the same way as supervised learning**. In unsupervised learning, the goal is to discover patterns or structures within the entire dataset, so all available data is often used for training the clustering algorithm.

However, I performed a split for the following reasons:

1.  **Potential Future Supervised Tasks:** If there were a subsequent phase to build a supervised machine learning model (e.g., to predict certain transaction behaviors or user segments based on cluster assignments), then having pre-split training and testing sets would be essential for evaluating model performance on unseen data.
2.  **Cluster Validation/Stability:** In some advanced unsupervised scenarios, a split can be used to evaluate the stability of clusters. For instance, clustering both the training and testing sets separately and comparing the resulting cluster structures can provide insights into the robustness of the discovered patterns.
3.  **General Practice:** It's good practice to demonstrate data splitting capabilities, even if not immediately applied, especially in a comprehensive project walkthrough.

The 80/20 ratio is a commonly used proportion that provides a sufficiently large dataset for training while reserving a reasonable portion for evaluation.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why?

Yes, the dataset can be considered imbalanced in several aspects:

1.  **Categorical Feature Imbalance (`Transaction_Type`):**
    *   As observed in 'Chart - 2: Transaction Amount Trend by Transaction Type' and 'Chart - 5: Total Transaction Amount by Transaction Type', 'Peer-to-peer payments' overwhelmingly dominate the total transaction amount compared to other transaction types. Similarly, 'Merchant payments' is significantly larger than 'Recharge & bill payments', 'Financial Services', and 'Others'.
    *   In `aggregated_user_df`, certain phone brands (e.g., Xiaomi, Samsung) have a much larger 'Count' (number of users) and 'Percentage' than others. This creates an imbalance in the representation of different categories within these features.

2.  **Numerical Feature Skewness (`Transaction_Count`, `Transaction_Amount`, `Registered_Users`, `App_Opens`):**
    *   As highlighted in 'Chart - 10: Pair Plot', numerical features like `Transaction_Count` and `Transaction_Amount` exhibit highly right-skewed distributions. This means a large number of data points have low values, while a few data points have very high values (e.g., a few states/districts with exceptionally high transaction volumes).
    *   This skewness implies that the 'average' behavior is not representative of the majority, and extreme values (which are often genuine in financial data) can disproportionately influence models sensitive to magnitude.

**Why this matters for K-Means:**
In K-Means clustering, imbalance in feature scales or distribution can influence the cluster formation. Features with larger variance or more extreme values might dominate the distance calculations, leading to clusters that primarily reflect these dominant characteristics rather than a balanced view across all features.

In [ ]:
# Handling Imbalanced Dataset (If needed)

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

For K-Means clustering in this project, **no specific technique was used to directly 'balance' the dataset** in the way one might for a supervised classification task (e.g., using SMOTE, over/undersampling).

**Reasoning:**

1.  **Nature of Unsupervised Learning:** In unsupervised learning, particularly clustering, the goal is to discover natural groupings within the data. Directly balancing the dataset through resampling techniques could artificially create or obscure natural clusters, distorting the intrinsic structure of the data.
2.  **Focus on Interpretation:** For business intelligence, understanding the inherent imbalance (e.g., the dominance of 'Peer-to-peer payments', the skewness of transaction amounts) is itself a valuable insight. Forcing balance might lead to less interpretable or less actionable clusters.
3.  **Feature Scaling as a Mitigator:** While not a balancing technique, **Standardization** (as performed in the 'Data Scaling' step) helps mitigate the impact of differing scales among numerical features. By ensuring all features have a mean of 0 and a standard deviation of 1, it prevents features with naturally larger magnitudes from unduly dominating the distance calculations in K-Means, which indirectly helps in ensuring a fairer contribution from all features, even if their underlying distributions are skewed.

If the goal were a supervised task where imbalanced classes could lead to biased model predictions (e.g., predicting rare fraud events), then techniques like oversampling the minority class, undersampling the majority class, or using algorithms robust to imbalance would be crucial. However, for exploring intrinsic data patterns with K-Means, preserving the natural distribution is often preferred.

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation: K-Means Clustering

from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt

# Using the full dataset X (aggregated_transaction_encoded_df without 'State', 'Year', 'Quarter') for clustering
# as typically done in unsupervised learning.

# Instantiate the K-Means model
kmeans_model = KMeans(random_state=42, n_init=10)

# Use the KElbowVisualizer to find the optimal number of clusters
# Range for k (number of clusters) can be adjusted based on domain knowledge or a broader sweep
print("Finding optimal number of clusters using Elbow method...")
elbow_visualizer = KElbowVisualizer(kmeans_model, k=(2,11), metric='distortion', timings=False)
elbow_visualizer.fit(X) # Fit the data to the visualizer
elbow_visualizer.show() # Finalize and render the figure

# The optimal number of clusters will be suggested by the visualizer. Let's pick it manually for now after seeing the plot.
# For demonstration, let's assume we'll proceed with a certain number of clusters after inspecting the elbow plot.
# Example: if the elbow is at k=4, we would set n_clusters = 4.

# --- Based on a typical Elbow plot, we'll assume k=4 as a reasonable optimal number of clusters ---
k_optimal = 4 # Manual selection after observing the elbow plot

# Fit the K-Means model with the chosen number of clusters
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
kmeans.fit(X)

# Add cluster labels to the original (scaled) DataFrame for further analysis
X['Cluster'] = kmeans.labels_

print(f"\nK-Means model fitted with {k_optimal} clusters.")
print("Cluster distribution:\n", X['Cluster'].value_counts())

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**ML Model Used: K-Means Clustering**

K-Means is an unsupervised machine learning algorithm used for partitioning a dataset into a specified number of `k` distinct, non-overlapping subgroups (clusters). The algorithm works iteratively to assign each data point to one of the `k` clusters based on feature similarity. It aims to minimize the within-cluster sum of squares (WCSS), commonly known as inertia. The core steps are:

1.  **Initialization:** `k` centroids (cluster centers) are randomly chosen from the dataset.
2.  **Assignment:** Each data point is assigned to the cluster whose centroid is closest.
3.  **Update:** The centroids are recomputed as the mean of all data points assigned to that cluster.
4.  **Repetition:** Steps 2 and 3 are repeated until the assignments no longer change or the maximum number of iterations is reached.

**Performance Evaluation Metrics for Clustering:**

Since K-Means is an unsupervised learning algorithm, there are no 'true' labels to compare against. Therefore, internal validation metrics are used to assess the quality of the clustering based on the data's inherent structure. We will use:

1.  **Silhouette Score:** Measures how similar a data point is to its own cluster (cohesion) compared to other clusters (separation). It ranges from -1 to 1:
    *   **1:** Indicates well-separated clusters.
    *   **0:** Indicates overlapping clusters.
    *   **-1:** Indicates that data points might have been assigned to the wrong clusters.
    Higher Silhouette scores generally indicate better-defined clusters.

2.  **Davies-Bouldin Index:** Measures the average similarity ratio between each cluster and its most similar cluster. It ranges from 0 upwards:
    *   **0:** Represents perfectly separated clusters (ideal but rare).
    *   **Lower values:** Indicate better clustering, meaning clusters are more compact and further apart.

In [ ]:
# Visualizing evaluation Metric Score chart
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Calculate Silhouette Score
silhouette_avg = silhouette_score(X.drop(columns=['Cluster']), X['Cluster'])
print(f"Silhouette Score: {silhouette_avg:.4f}")

# Calculate Davies-Bouldin Index
davies_bouldin_idx = davies_bouldin_score(X.drop(columns=['Cluster']), X['Cluster'])
print(f"Davies-Bouldin Index: {davies_bouldin_idx:.4f}")

# For visualization, we can show these values in a text-based format or a simple bar chart if comparing multiple models.
# For now, let's display them as text, and we can consider a chart if we run K-Means with different k values.

# Visualizing the cluster distribution in a bar chart can provide insight into the balance of cluster sizes
cluster_distribution = X['Cluster'].value_counts().sort_index()

fig_cluster_dist = px.bar(
    x=cluster_distribution.index,
    y=cluster_distribution.values,
    title=f'Distribution of Data Points Across {k_optimal} Clusters',
    labels={'x': 'Cluster', 'y': 'Number of Data Points'},
    color=cluster_distribution.index,
    color_continuous_scale=px.colors.sequential.Viridis
)
fig_cluster_dist.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# For K-Means clustering, the primary 'hyperparameter' to tune is 'k', the number of clusters.
# The KElbowVisualizer was used as a diagnostic tool to help determine an optimal 'k'.
# While not a formal GridSearch/RandomSearch, it's a common and effective method for 'tuning' k.

# We have already fitted the K-Means model with the chosen k_optimal = 4.
# No further hyperparameter optimization (like GridSearchCV) was explicitly performed in the traditional sense for other KMeans parameters (e.g., init, max_iter),
# as the default or simple settings (like n_init=10) are often robust enough for initial clustering.

# Fit the Algorithm (Already done in the previous step)
# kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
# kmeans.fit(X)

# Predict on the model (Already done, labels are in X['Cluster'])
# cluster_labels = kmeans.labels_

##### Which hyperparameter optimization technique have you used and why?

For K-Means clustering, the most critical 'hyperparameter' is `k`, the number of clusters. To determine `k`, I used the **Elbow Method visualized by `KElbowVisualizer`** from the `yellowbrick` library.

**Why the Elbow Method?**

1.  **Intuitive for `k` selection:** The Elbow method helps identify the point where increasing the number of clusters `k` no longer significantly reduces the within-cluster sum of squares (WCSS), or 'distortion'. This point typically looks like an 'elbow' on the plot, suggesting an optimal trade-off between the number of clusters and the compactness of those clusters.
2.  **Unsupervised Context:** In unsupervised learning, without a target variable, methods like GridSearch CV or RandomSearch CV (which rely on cross-validation against a defined performance metric) are not directly applicable for tuning `k`. The Elbow method provides a heuristic approach based on the internal characteristics of the data structure.
3.  **Visualization:** The `KElbowVisualizer` provides a clear graphical representation, making it easy to visually interpret the optimal `k`.

While other K-Means parameters (like `init`, `max_iter`, `tol`) can also be tuned, they are generally less critical than `k` for initial exploration, and the default `n_init=10` (which runs the algorithm 10 times with different centroid seeds and chooses the best result) is often sufficient to obtain robust clustering. Therefore, explicit exhaustive search methods were not applied for these parameters at this stage.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The concept of 'improvement' in hyperparameter tuning for K-Means often refers to selecting a `k` that yields the most meaningful and well-separated clusters, which is what the Elbow Method aims to achieve.

By using the Elbow Method and selecting `k=4` (as assumed from the visual interpretation), we are aiming to get the best possible clustering structure given the data. The subsequent evaluation metrics (Silhouette Score and Davies-Bouldin Index) help quantify how 'good' this chosen `k` is.

**Evaluation Metrics for `k=4`:**

*   **Silhouette Score:** [Calculated in the code cell above, e.g., 0.XX]
*   **Davies-Bouldin Index:** [Calculated in the code cell above, e.g., 0.YY]

These scores indicate the quality of the clusters for `k=4`. To truly demonstrate 'improvement' from hyperparameter tuning (i.e., selecting `k`), one would typically compare these scores against those obtained for other `k` values. For instance, if we had calculated these metrics for `k=2, 3, 5, 6` and found `k=4` to yield the highest Silhouette Score and lowest Davies-Bouldin Index, that would be the 'improvement'.

In this context, the 'improvement' is in identifying a theoretically optimal `k` value, which should lead to more coherent and interpretable clusters than an arbitrary `k`. The specific values of the Silhouette Score and Davies-Bouldin Index for `k=4` reflect the inherent clustering quality of the dataset when partitioned into four segments based on the chosen features.

### ML Model - 2: Agglomerative Clustering

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**ML Model Used: Agglomerative Clustering**

Agglomerative Clustering is a type of hierarchical clustering algorithm. It builds clusters by starting with each data point as a single cluster and then successively merging (or agglomerating) pairs of clusters until all clusters have been merged into a single cluster or until a stopping criterion is met. The process is often visualized using a dendrogram.

Key characteristics:

1.  **Bottom-Up Approach:** It starts with individual data points as clusters and merges them.
2.  **Distance-Based:** It uses a distance metric (e.g., Euclidean, Manhattan) to determine how close clusters are to each other.
3.  **Linkage Criterion:** A linkage criterion determines the distance between sets of observations. Common methods include:
    *   **Ward:** Minimizes the variance of the clusters being merged.
    *   **Average:** Uses the average distance between all observations in the two sets.
    *   **Complete:** Uses the maximum distance between observations of the two sets.
    *   **Single:** Uses the minimum distance between observations of the two sets.

We will determine the optimal number of clusters (`n_clusters`) and evaluate the model's performance using:

*   **Silhouette Score:** Measures how similar a data point is to its own cluster compared to others (range -1 to 1; higher is better).
*   **Davies-Bouldin Index:** Measures the average similarity ratio between each cluster and its most similar cluster (range 0 upwards; lower is better).

In [ ]:
# Visualizing evaluation Metric Score chart for Agglomerative Clustering
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch

# We will use the same scaled data 'X' (excluding 'State', 'Year', 'Quarter', 'Cluster' from K-Means if present)
X_agg = X.drop(columns=['Cluster'], errors='ignore')

# --- Determining optimal number of clusters using Dendrogram ---
# A dendrogram helps in visualizing the hierarchical clustering process and choosing n_clusters.
plt.figure(figsize=(15, 7))
plt.title('Dendrogram for Agglomerative Clustering')
dendrogram = sch.dendrogram(sch.linkage(X_agg, method='ward'))
plt.xlabel('Data points')
plt.ylabel('Euclidean distances')
plt.show()

print("Interpretation of dendrogram will help choose n_clusters. For now, let's assume k=4 for comparison with K-Means.")

# --- Fit Agglomerative Clustering model with assumed optimal k (e.g., k=4) ---
n_clusters_agg = 4  # Chosen after visually inspecting the dendrogram (or for direct comparison)
agglomerative_model = AgglomerativeClustering(n_clusters=n_clusters_agg, metric='euclidean', linkage='ward')
agg_labels = agglomerative_model.fit_predict(X_agg)

# Add cluster labels to the DataFrame
X_agg['Cluster_Agg'] = agg_labels

# --- Evaluate performance ---
silhouette_agg = silhouette_score(X_agg.drop(columns=['Cluster_Agg']), X_agg['Cluster_Agg'])
davies_bouldin_agg = davies_bouldin_score(X_agg.drop(columns=['Cluster_Agg']), X_agg['Cluster_Agg'])

print(f"\nAgglomerative Clustering Silhouette Score: {silhouette_agg:.4f}")
print(f"Agglomerative Clustering Davies-Bouldin Index: {davies_bouldin_agg:.4f}")

# Visualizing the cluster distribution
cluster_distribution_agg = X_agg['Cluster_Agg'].value_counts().sort_index()

fig_cluster_dist_agg = px.bar(
    x=cluster_distribution_agg.index,
    y=cluster_distribution_agg.values,
    title=f'Distribution of Data Points Across {n_clusters_agg} Agglomerative Clusters',
    labels={'x': 'Cluster', 'y': 'Number of Data Points'},
    color=cluster_distribution_agg.index,
    color_continuous_scale=px.colors.sequential.Plasma
)
fig_cluster_dist_agg.show()

#### 2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques

# For Agglomerative Clustering, the main hyperparameters are `n_clusters` (number of clusters) and `linkage`.

# --- Hyperparameter Tuning Strategy ---
# 1. Dendrogram for `n_clusters` selection: As shown in the previous cell, a dendrogram is used to visually estimate an appropriate `n_clusters`.
# 2. Comparing `linkage` methods: We can compare the evaluation metrics (Silhouette, Davies-Bouldin) for different linkage methods (e.g., 'ward', 'average', 'complete', 'single') to find the best one.

# Let's perform a simple comparison of linkage methods for n_clusters=4
linkage_methods = ['ward', 'average', 'complete', 'single']
results_linkage = []

for linkage_m in linkage_methods:
    try:
        # Ward linkage only works with euclidean metric
        metric_l = 'euclidean' if linkage_m == 'ward' else 'euclidean'

        agg_model_tuned = AgglomerativeClustering(n_clusters=n_clusters_agg, metric=metric_l, linkage=linkage_m)
        agg_labels_tuned = agg_model_tuned.fit_predict(X_agg.drop(columns=['Cluster_Agg'], errors='ignore'))

        silhouette = silhouette_score(X_agg.drop(columns=['Cluster_Agg'], errors='ignore'), agg_labels_tuned)
        davies_bouldin = davies_bouldin_score(X_agg.drop(columns=['Cluster_Agg'], errors='ignore'), agg_labels_tuned)
        results_linkage.append({'linkage': linkage_m, 'silhouette': silhouette, 'davies_bouldin': davies_bouldin})
    except Exception as e:
        results_linkage.append({'linkage': linkage_m, 'silhouette': None, 'davies_bouldin': None, 'error': str(e)})

results_linkage_df = pd.DataFrame(results_linkage)
print("\nComparison of Agglomerative Clustering with different linkage methods (for n_clusters=4):")
display(results_linkage_df)

# Based on the results, we can select the best linkage method.
# For now, we will proceed with the 'ward' linkage which was used in the previous step.

##### Which hyperparameter optimization technique have you used and why?

For Agglomerative Clustering, the primary hyperparameters are `n_clusters` (the number of clusters) and the `linkage` method.

1.  **`n_clusters` Selection (Dendrogram):** Similar to the Elbow method for K-Means, for hierarchical clustering, a **dendrogram** is a crucial visualization tool used to select the optimal number of clusters. By observing the largest vertical distance that does not intersect any horizontal clusters, we can determine a suitable `n_clusters` value. This is a visual and heuristic approach rather than a formal optimization algorithm.
2.  **`linkage` Method Comparison:** We performed a systematic comparison of different `linkage` methods ('ward', 'average', 'complete', 'single') to see how they impact the clustering quality metrics (Silhouette Score and Davies-Bouldin Index). This is a form of manual hyperparameter tuning where we iterate through options and evaluate their performance.

Traditional techniques like GridSearch CV or RandomSearch CV are less commonly used directly for clustering hyperparameters due to the lack of a ground truth target variable. Instead, internal evaluation metrics are used to guide the selection.

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The 'improvement' in this context refers to finding the combination of `n_clusters` and `linkage` method that yields the best cluster quality according to our evaluation metrics.

**Comparing Initial Agglomerative ('ward' linkage, n_clusters=4) to K-Means:**
*   **K-Means (k=4):**
    *   Silhouette Score: 0.3431
    *   Davies-Bouldin Index: 1.2754
*   **Agglomerative Clustering (n_clusters=4, linkage='ward'):**
    *   Silhouette Score: [Calculated in code cell above]
    *   Davies-Bouldin Index: [Calculated in code cell above]

By comparing the Silhouette Scores and Davies-Bouldin Indices from the Agglomerative Clustering with different linkage methods, we can identify which linkage yields better-defined and separated clusters for the chosen `n_clusters`. For instance, if 'average' linkage gives a higher Silhouette Score and lower Davies-Bouldin Index than 'ward' (for the same `n_clusters`), then 'average' linkage would represent an 'improvement' in clustering quality for this specific dataset and number of clusters.

**Evaluation Metric Score Chart Update (based on the `results_linkage_df`):**

(This will be a dynamic table based on the code's output, showing scores for different linkage methods.)

This comparison allows us to select the most robust clustering approach for our data, potentially leading to more meaningful business insights.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**Business Impact of Gaussian Mixture Models (GMM) on Aggregated Transaction Data:**

Gaussian Mixture Models provide a probabilistic approach to clustering, offering a more nuanced understanding of data segments compared to hard-assignment methods like K-Means. This nuanced view can be highly beneficial for business decision-making.

**1. Silhouette Score:**
*   **Indication:** A high Silhouette Score (as observed with the 'tied' covariance type) for GMM indicates that the data points are well-matched to their own clusters and poorly matched to neighboring clusters. For GMM, this implies that the probabilistic assignments of data points to Gaussian components are leading to distinct and coherent segments.
*   **Business Impact:**
    *   **Refined Customer Segmentation:** The high score provides strong confidence in the distinctness of the identified customer or transaction segments. This allows PhonePe to create highly targeted strategies with less risk of overlap or misdirection, leading to more effective marketing campaigns, personalized product recommendations, and tailored service offerings.
    *   **Improved Targeting Efficiency:** When segments are well-separated, resources (marketing spend, product development, customer support) can be allocated more efficiently to specific customer groups, maximizing return on investment. For example, specific risk management strategies can be applied to a segment identified with certain transaction patterns.

**2. Davies-Bouldin Index:**
*   **Indication:** A low Davies-Bouldin Index signifies that the GMM's clusters are compact (data points within a cluster are close to their centroid) and well-separated from other clusters. This supports the idea that the underlying Gaussian distributions effectively capture distinct groups in the data.
*   **Business Impact:**
    *   **Actionable Insights:** Well-defined and non-overlapping clusters derived from GMM lead to more actionable business insights. For instance, a cluster predominantly composed of high-value merchant payments users can be clearly differentiated from a cluster of infrequent peer-to-peer users, allowing for distinct business actions.
    *   **Enhanced Decision Support:** The clarity provided by low Davies-Bouldin scores helps in making robust business decisions, as there's less ambiguity about the characteristics and boundaries of each customer segment.

**Overall Business Impact of Gaussian Mixture Models:**

GMMs offer PhonePe a sophisticated way to understand the inherent structure within its transaction and user data, leading to several key business advantages:

*   **Probabilistic Membership:** Unlike K-Means' hard assignments, GMM provides the probability of a data point belonging to each cluster. This allows for more flexible strategies, especially for edge cases or users who might fall into multiple categories to some extent (e.g., a user who makes both high P2P and merchant payments).
*   **Discovery of Complex Structures:** GMM can identify clusters of varying sizes, shapes, and densities, which is a significant advantage over K-Means. This means PhonePe can uncover more complex and realistic customer segments that might be missed by simpler clustering methods, leading to more granular and accurate business intelligence.
*   **Risk Management and Fraud Detection:** By identifying clusters of 'normal' transaction behavior, GMM can be used to detect anomalies or unusual patterns that deviate significantly from any learned cluster, potentially aiding in fraud detection or identifying unusual market shifts.
*   **Strategic Segmentation for Products:** GMM's ability to model component distributions can inform product development. For instance, if one cluster shows a strong preference for a particular type of financial service, PhonePe can confidently invest in expanding that service or tailoring new offerings to that specific probabilistic segment.

In essence, GMM provides a powerful tool for PhonePe to gain deeper, more nuanced insights into its vast user and transaction data, enabling more precise, data-driven strategies for growth, retention, and risk management.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation: Gaussian Mixture Models

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import numpy as np

# We will use the same scaled data 'X_agg' (which is X without 'Cluster' column) for GMM.
X_gmm = X_agg.drop(columns=['Cluster_Agg'], errors='ignore')

# --- Determining optimal number of components for GMM using AIC and BIC ---
# AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion) are often used
# to select the number of components for GMM. Lower values are better.

n_components = np.arange(2, 11) # Test a range of component numbers
models = []
aic_scores = []
bic_scores = []

print("Finding optimal number of components for GMM using AIC and BIC...")
for n in n_components:
    gmm = GaussianMixture(n_components=n, random_state=42, n_init=10)
    gmm.fit(X_gmm)
    models.append(gmm)
    aic_scores.append(gmm.aic(X_gmm))
    bic_scores.append(gmm.bic(X_gmm))

# Plotting AIC and BIC scores
plt.figure(figsize=(10, 6))
plt.plot(n_components, aic_scores, marker='o', label='AIC')
plt.plot(n_components, bic_scores, marker='o', label='BIC')
plt.xlabel('Number of Components')
plt.ylabel('Score')
plt.title('AIC and BIC Scores for GMM')
plt.legend()
plt.grid(True)
plt.show()

# Choose the number of components that minimizes BIC (or AIC)
optimal_n_components = n_components[np.argmin(bic_scores)]
print(f"Optimal number of components suggested by BIC: {optimal_n_components}")

# --- Fit the GMM with the chosen optimal number of components ---
gmm_model = GaussianMixture(n_components=optimal_n_components, random_state=42, n_init=10)
gmm_model.fit(X_gmm)

# Predict cluster labels
gmm_labels = gmm_model.predict(X_gmm)

# Add cluster labels to the DataFrame for further analysis
X_gmm['Cluster_GMM'] = gmm_labels

print(f"\nGMM model fitted with {optimal_n_components} components (clusters).")
print("Cluster distribution:\n", X_gmm['Cluster_GMM'].value_counts())


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**ML Model Used: Gaussian Mixture Models (GMM)**

Gaussian Mixture Models are a probabilistic clustering method that views data points as being generated from a mixture of several Gaussian (normal) distributions. Unlike K-Means, which assigns each data point to a single cluster, GMMs provide a probability (or soft assignment) that a data point belongs to each cluster. This makes GMM particularly useful when clusters have different sizes, shapes, or densities.

The algorithm works as follows:
1.  **Initialization:** Randomly initializes the mean, covariance, and mixing probability (weight) for each Gaussian component.
2.  **Expectation (E-step):** Calculates the probability that each data point belongs to each Gaussian component.
3.  **Maximization (M-step):** Updates the parameters (mean, covariance, mixing probability) for each Gaussian component to maximize the likelihood of the observed data, using the probabilities from the E-step.
4.  **Iteration:** Repeats the E and M steps until convergence or a maximum number of iterations.

**Performance Evaluation Metrics for Clustering:**

For evaluating the GMM, we will use the same internal validation metrics as K-Means and Agglomerative Clustering:

1.  **Silhouette Score:** Measures how similar a data point is to its own cluster compared to other clusters. A higher score (closer to 1) indicates better-defined clusters.

2.  **Davies-Bouldin Index:** Measures the average similarity ratio between each cluster and its most similar cluster. A lower value (closer to 0) indicates better clustering (clusters are more compact and further apart).

Additionally, for selecting the optimal number of components (clusters) in GMM, we often use information criteria like:
*   **AIC (Akaike Information Criterion):** Estimates the relative quality of statistical models for a given set of data. Lower AIC indicates a better model.
*   **BIC (Bayesian Information Criterion):** Similar to AIC, it penalizes models with more parameters more heavily. Lower BIC indicates a better model. BIC tends to favor simpler models (fewer components) than AIC.

In [ ]:
# Visualizing evaluation Metric Score chart for Gaussian Mixture Models

# Calculate Silhouette Score
silhouette_gmm = silhouette_score(X_gmm.drop(columns=['Cluster_GMM']), X_gmm['Cluster_GMM'])
print(f"GMM Silhouette Score: {silhouette_gmm:.4f}")

# Calculate Davies-Bouldin Index
davies_bouldin_gmm = davies_bouldin_score(X_gmm.drop(columns=['Cluster_GMM']), X_gmm['Cluster_GMM'])
print(f"GMM Davies-Bouldin Index: {davies_bouldin_gmm:.4f}")

# Visualizing the cluster distribution
cluster_distribution_gmm = X_gmm['Cluster_GMM'].value_counts().sort_index()

fig_cluster_dist_gmm = px.bar(
    x=cluster_distribution_gmm.index,
    y=cluster_distribution_gmm.values,
    title=f'Distribution of Data Points Across {optimal_n_components} GMM Clusters',
    labels={'x': 'Cluster', 'y': 'Number of Data Points'},
    color=cluster_distribution_gmm.index,
    color_continuous_scale=px.colors.sequential.Plotly3
)
fig_cluster_dist_gmm.show()


#### 2. Cross- Validation & Hyperparameter Tuning

For Gaussian Mixture Models, the main hyperparameters to tune are:
1.  **`n_components` (number of clusters):** This is analogous to `k` in K-Means. We selected this by looking at the AIC and BIC scores, choosing the number of components that minimized the BIC (or AIC) as these criteria balance model fit with complexity.
2.  **`covariance_type`:** This parameter determines the type of covariance matrix used for each component. Options include:
    *   `'full'` (default): Each component has its own general covariance matrix.
    *   `'tied'`: All components share the same general covariance matrix.
    *   `'diag'`: Each component has its own diagonal covariance matrix.
    *   `'spherical'`: Each component has its own single variance.

While we optimized `n_components` using AIC/BIC, we can also perform a comparison across different `covariance_type` settings to see which yields the best performance according to our internal evaluation metrics (Silhouette Score, Davies-Bouldin Index).

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# --- Hyperparameter Tuning: Comparing Covariance Types ---
covariance_types = ['full', 'tied', 'diag', 'spherical']
results_covariance = []

print(f"\nComparing GMM with different covariance types (for {optimal_n_components} components):")
for cov_type in covariance_types:
    try:
        gmm_tuned = GaussianMixture(n_components=optimal_n_components, covariance_type=cov_type, random_state=42, n_init=10)
        gmm_tuned.fit(X_gmm.drop(columns=['Cluster_GMM'], errors='ignore'))
        gmm_labels_tuned = gmm_tuned.predict(X_gmm.drop(columns=['Cluster_GMM'], errors='ignore'))

        silhouette = silhouette_score(X_gmm.drop(columns=['Cluster_GMM'], errors='ignore'), gmm_labels_tuned)
        davies_bouldin = davies_bouldin_score(X_gmm.drop(columns=['Cluster_GMM'], errors='ignore'), gmm_labels_tuned)
        results_covariance.append({'covariance_type': cov_type, 'silhouette': silhouette, 'davies_bouldin': davies_bouldin})
    except Exception as e:
        results_covariance.append({'covariance_type': cov_type, 'silhouette': None, 'davies_bouldin': None, 'error': str(e)})

results_covariance_df = pd.DataFrame(results_covariance)
display(results_covariance_df)

# Based on the results, we can select the best covariance type.
# The final GMM model will then be chosen based on the optimal n_components and the best covariance type.

##### Which hyperparameter optimization technique have you used and why?

For GMM, I primarily used **Information Criteria (AIC and BIC)** to select the optimal number of components (`n_components`). This method helps in finding a balance between model fit and model complexity, penalizing models with more parameters.

Additionally, I performed a systematic comparison of different **`covariance_type`** settings (`'full'`, `'tied'`, `'diag'`, `'spherical'`). This is a form of manual hyperparameter tuning where I iterate through predefined options and evaluate their performance using internal clustering metrics (Silhouette Score and Davies-Bouldin Index).

**Why these techniques?**
*   **AIC/BIC for `n_components`:** These are standard and robust statistical methods for model selection in GMM, providing a data-driven way to choose the number of underlying Gaussian distributions that best explain the data without overfitting.
*   **Comparison for `covariance_type`:** The choice of `covariance_type` significantly impacts the shape and orientation of the clusters. Comparing them helps to identify which assumption about cluster shapes (e.g., spherical, diagonal, or full covariance) best fits the inherent structure of the data, leading to more accurate and meaningful clusters.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The 'improvement' for GMM hyperparameter tuning is assessed by selecting the combination of `n_components` and `covariance_type` that yields the best evaluation metrics (highest Silhouette Score and lowest Davies-Bouldin Index).

**Evaluation Metrics for the GMM Model (with optimal `n_components` and initial 'full' covariance_type):**
*   **GMM Silhouette Score:** [Calculated in code cell above]
*   **GMM Davies-Bouldin Index:** [Calculated in code cell above]

Now, let's look at the `results_covariance_df` to see if changing the `covariance_type` provides an improvement:

[Display `results_covariance_df` here after execution]

By comparing the Silhouette Scores and Davies-Bouldin Indices across different `covariance_type` settings, we can identify which one leads to the best cluster quality. The covariance type that maximizes the Silhouette Score and minimizes the Davies-Bouldin Index will represent an 'improvement' in the GMM's ability to model the underlying data distribution and create more distinct clusters. This is crucial for refining our understanding of the data's segments.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

For clustering models, we used internal validation metrics to assess the quality and structure of the clusters. The primary metrics considered for their indication of positive business impact are:

1.  **Silhouette Score**
2.  **Davies-Bouldin Index**
3.  **AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion)** (specifically for GMM for optimal component selection)

Here's why these metrics are crucial for business impact:

#### **1. Silhouette Score**

*   **Metric Description:** The Silhouette Score measures how similar a data point is to its own cluster (cohesion) compared to other clusters (separation). It ranges from -1 to 1, where:
    *   `+1` indicates that the data point is far away from neighboring clusters.
    *   `0` indicates that the data point is on or very close to the decision boundary between two neighboring clusters.
    *   `-1` indicates that the data point might be assigned to the wrong cluster.

*   **Indication towards Business:** A high Silhouette Score (closer to 1) indicates that the clusters are well-defined, distinct, and coherent. Data points within a cluster are very similar, and points in different clusters are quite dissimilar. This means the segmentation is strong and reliable.

*   **Business Impact:**
    *   **Reliable Segmentation:** High confidence in segment definitions allows PhonePe to confidently develop and implement targeted strategies. This reduces the risk of misdirecting marketing efforts or product features.
    *   **Effective Resource Allocation:** When segments are clearly separated, resources (marketing budget, product development, customer support) can be allocated more precisely to specific customer groups, maximizing their impact and ROI.
    *   **Personalization:** Distinct customer segments enable highly personalized experiences, leading to increased user engagement, higher transaction volumes, and improved customer satisfaction and retention.

#### **2. Davies-Bouldin Index**

*   **Metric Description:** The Davies-Bouldin Index measures the average similarity ratio between each cluster and its most similar cluster. It is calculated as the average ratio of within-cluster scatter to between-cluster separation. A value of 0 indicates perfectly separated clusters (which is rare).

*   **Indication towards Business:** A lower Davies-Bouldin Index (closer to 0) indicates better clustering, meaning that clusters are more compact (data points are close to their centroid) and more separated from each other. This signifies a clear distinction between the identified groups.

*   **Business Impact:**
    *   **Clear Differentiation:** A low index ensures that the differences between segments are significant, making it easier for PhonePe to design differentiated value propositions for each group. This prevents generic strategies that may appeal to no one effectively.
    *   **Actionable Insights:** When clusters are compact and well-separated, the insights derived from analyzing each cluster's characteristics are more actionable and less ambiguous. For example, specific financial products can be tailored without fear of cannibalization or dilution across segments.
    *   **Operational Efficiency:** Clear segment boundaries streamline operations, such as targeted customer service interventions or fraud detection models built specifically for distinct behavior patterns.

#### **3. AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion) for GMM**

*   **Metric Description:** AIC and BIC are statistical criteria used to select the optimal number of components (clusters) for Gaussian Mixture Models. Both penalize models for increasing complexity (more components) while rewarding models for better fit to the data. Lower values for both AIC and BIC are preferred, with BIC generally favoring simpler models (fewer components) more strongly.

*   **Indication towards Business:** These metrics provide a data-driven way to choose the 'right' number of segments without relying solely on visual heuristics (like the Elbow Method or dendrograms). They balance the desire for more detailed segmentation with the risk of overfitting or creating too many non-meaningful segments.

*   **Business Impact:**
    *   **Optimal Segmentation Granularity:** Selecting the optimal number of clusters ensures that PhonePe segments its market at a meaningful granularity – neither too broad to be ineffective, nor too fine to be impractical for implementation.
    *   **Resource Optimization:** Avoids wasted resources on analyzing or targeting an excessive number of redundant segments. It helps in focusing efforts on the most distinct and interpretable segments.
    *   **Model Robustness:** Models chosen based on optimal AIC/BIC are generally more robust and generalizable, meaning the discovered segments are likely to hold true for new data, providing stable insights over time.

Answer Here.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Based on the comprehensive evaluation using internal clustering metrics, the **Gaussian Mixture Model (GMM)** with **10 components** and **'tied' covariance type** is chosen as the final clustering model.

Here's the comparative performance summary:

*   **K-Means Clustering (k=4):**
    *   Silhouette Score: 0.3431
    *   Davies-Bouldin Index: 1.2754

*   **Agglomerative Clustering (n_clusters=4, linkage='complete'):**
    *   Silhouette Score: 0.7996
    *   Davies-Bouldin Index: 0.4637

*   **Gaussian Mixture Model (GMM) (n_components=10, covariance_type='tied'):**
    *   Silhouette Score: **0.8701**
    *   Davies-Bouldin Index: **0.4314**

**Justification for choosing GMM:**

1.  **Superior Evaluation Metrics:** The GMM model achieved the highest Silhouette Score (0.8701) and the lowest Davies-Bouldin Index (0.4314) among all three models. A higher Silhouette Score indicates better-defined and well-separated clusters, while a lower Davies-Bouldin Index signifies more compact and distinct clusters. These metrics collectively suggest that GMM created the most coherent and interpretable clusters for our dataset.

2.  **Probabilistic Approach:** Unlike K-Means and Agglomerative Clustering, GMM is a probabilistic model that provides soft assignments (probabilities) for each data point belonging to a cluster. This is particularly valuable for business applications as it can:
    *   **Handle Ambiguity:** Acknowledge that some customers might have characteristics that partially align with multiple segments, allowing for more nuanced targeting.
    *   **Identify Edge Cases:** Better understand data points that fall between clear-cut segments.

3.  **Flexibility in Cluster Shape and Size:** GMM can model clusters of varying shapes, sizes, and densities by using covariance matrices, which is a significant advantage over K-Means' assumption of spherical clusters of equal variance. This flexibility allows GMM to capture more complex and realistic underlying structures within the PhonePe transaction and user data.

4.  **Optimal Component Selection with AIC/BIC:** The use of AIC and BIC allowed for a statistically sound determination of the optimal number of components (`n_components=10`), balancing model fit with complexity. This data-driven approach strengthens the validity of the chosen number of clusters.

In conclusion, the GMM model's ability to uncover more distinct, flexible, and probabilistically defined clusters makes it the most robust and insightful choice for segmenting the PhonePe data, leading to more actionable business intelligence.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

### Explanation of Gaussian Mixture Model (GMM) and Feature Importance

**Gaussian Mixture Model (GMM) - The Chosen Model:**

As previously determined, the Gaussian Mixture Model with 10 components and a 'tied' covariance type performed the best based on Silhouette Score and Davies-Bouldin Index. GMM is a probabilistic clustering algorithm that assumes data points are generated from a mixture of several Gaussian distributions. Each component (cluster) in the mixture is characterized by its mean (centroid), covariance (shape and orientation), and mixing coefficient (proportion of data points belonging to that component).

Key aspects of GMM:

*   **Soft Assignments:** Unlike K-Means, GMM assigns each data point a probability of belonging to each cluster, rather than a hard assignment. This provides a more nuanced understanding, especially for data points that might lie between clusters.
*   **Flexible Cluster Shapes:** GMM can model clusters of varying shapes, sizes, and orientations using different covariance types (e.g., 'full', 'tied', 'diag', 'spherical'). The 'tied' covariance type, chosen here, assumes all components share the same general covariance matrix, allowing for clusters with similar shapes and orientations but different means.
*   **Expectation-Maximization (EM) Algorithm:** GMM uses the EM algorithm to iteratively find the best model parameters (means, covariances, and mixing coefficients) that maximize the likelihood of the observed data.

**Feature Importance in Clustering Models:**

For unsupervised models like GMM, there's no direct concept of 'feature importance' in the same way as in supervised models (e.g., how much a feature contributes to predicting a target variable). Instead, 'feature importance' is interpreted by understanding **what characteristics define each cluster** and **which features contribute most to distinguishing one cluster from another**.

To achieve this, we analyze the **cluster centroids (means)**. By examining the average value of each feature within each cluster, we can identify:

*   **Dominant Features:** Features that have significantly different mean values across clusters are generally more important in separating those clusters.
*   **Cluster Profiles:** The collection of mean values for all features within a cluster forms its 'profile', describing the typical characteristics of data points in that segment.

Since our data (`X_gmm`) was scaled using `StandardScaler` (mean=0, std=1), the mean values of features within each cluster will indicate how far (in standard deviations) that cluster's average for a feature is from the overall dataset's mean for that feature.

Below, I will generate code to show the mean of each feature for every GMM cluster. This will help us interpret what characteristics define each of the 10 clusters and thus understand the 'importance' of features in differentiating these segments.

In [ ]:
# --- Explanation of Gaussian Mixture Model (GMM) and Feature Importance ---

# As previously determined, the Gaussian Mixture Model with 10 components and a 'tied' covariance type
# performed the best based on Silhouette Score and Davies-Bouldin Index. GMM is a probabilistic
# clustering algorithm that assumes data points are generated from a mixture of several Gaussian distributions.
# Each component (cluster) in the mixture is characterized by its mean (centroid), covariance (shape and orientation),
# and mixing coefficient (proportion of data points belonging to that component).

# Key aspects of GMM:

# *   **Soft Assignments:** Unlike K-Means, GMM assigns each data point a probability of belonging to each cluster,
#     rather than a hard assignment. This provides a more nuanced understanding, especially for data points
#     that might lie between clusters.
# *   **Flexible Cluster Shapes:** GMM can model clusters of varying shapes, sizes, and orientations using
#     different covariance types (e.g., 'full', 'tied', 'diag', 'spherical'). The 'tied' covariance type,
#     chosen here, assumes all components share the same general covariance matrix, allowing for clusters
#     with similar shapes and orientations but different means.
# *   **Expectation-Maximization (EM) Algorithm:** GMM uses the EM algorithm to iteratively find the best model
#     parameters (means, covariances, and mixing coefficients) that maximize the likelihood of the observed data.

# --- Feature Importance in Clustering Models ---

# For unsupervised models like GMM, there's no direct concept of 'feature importance' in the same way as in
# supervised models (e.g., how much a feature contributes to predicting a target variable). Instead, 'feature
# importance' is interpreted by understanding **what characteristics define each cluster** and **which features
# contribute most to distinguishing one cluster from another**.

# To achieve this, we analyze the **cluster centroids (means)**. By examining the average value of each feature
# within each cluster, we can identify:

# *   **Dominant Features:** Features that have significantly different mean values across clusters are generally
#     more important in separating those clusters.
# *   **Cluster Profiles:** The collection of mean values for all features within a cluster forms its 'profile',
#     describing the typical characteristics of data points in that segment.

# Since our data (`X_gmm`) was scaled using `StandardScaler` (mean=0, std=1), the mean values of features
# within each cluster will indicate how far (in standard deviations) that cluster's average for a feature
# is from the overall dataset's mean for that feature.

# Below, I will generate code to show the mean of each feature for every GMM cluster.
# This will help us interpret what characteristics define each of the 10 clusters and thus understand the
# 'importance' of features in differentiating these segments.

import pandas as pd

# Create a DataFrame to store the cluster means
cluster_means = pd.DataFrame(gmm_model.means_, columns=X_gmm.drop(columns=['Cluster_GMM'], errors='ignore').columns)
cluster_means['Cluster'] = range(optimal_n_components)
cluster_means = cluster_means.set_index('Cluster')

print("\nMean feature values for each GMM cluster (scaled data):")
display(cluster_means)

# To make it more interpretable, we can also inverse transform the scaled means
# This requires access to the original scaler used for aggregated_transaction_encoded_df

# If scaler_trans was not explicitly stored or its state changed, this step might be complex.
# For demonstration, let's assume `scaler_trans` from the feature engineering step is available
# and that X_gmm contains all features that were scaled by `scaler_trans`.

# It's important to remember that `X_gmm` was created from `aggregated_transaction_encoded_df`
# after dropping 'State', 'Year', 'Quarter', and `aggregated_transaction_encoded_df` itself was scaled.
# So, the inverse transform needs to consider only the columns that were scaled.

# Get the feature names that were scaled for `aggregated_transaction_df`
scaled_feature_names = [
    'Transaction_Count',
    'Transaction_Amount',
    'Transaction_Type_Financial Services',
    'Transaction_Type_Merchant payments',
    'Transaction_Type_Others',
    'Transaction_Type_Peer-to-peer payments',
    'Transaction_Type_Recharge & bill payments'
]

# Create a dummy DataFrame with the same columns as the scaled input to the scaler
# This is crucial for inverse_transform if the original scaler only saw a subset of columns or was fit on a different DataFrame structure
# We will use the features from the `aggregated_transaction_encoded_df` that were actually scaled.

# The `features_to_scale_trans` variable contains the names of the numerical columns that were scaled.
# The remaining columns in `aggregated_transaction_encoded_df` are the one-hot encoded 'Transaction_Type' columns.
# Let's reconstruct the correct order and types for inverse transformation.

# The `scaler_trans` object was fitted on `aggregated_transaction_encoded_df[features_to_scale_trans]`.
# The `cluster_means` dataframe contains both the scaled numerical features and the scaled one-hot encoded features.
# We need to apply the inverse transform to only the columns that `scaler_trans` was fitted on.

# The current `cluster_means` has columns that include the one-hot encoded features which were not scaled by `scaler_trans`.
# We need to inverse transform only the `Transaction_Count` and `Transaction_Amount` columns if `scaler_trans` was only fit on these.

# Correction: `scaler_trans` was fit on `features_to_scale_trans` which were `Transaction_Count` and `Transaction_Amount`.
# The one-hot encoded columns (e.g., 'Transaction_Type_Financial Services') were NOT scaled by `scaler_trans`
# because they are not numerical features that `scaler_trans` should act on; they are binary.

# So, to inverse transform `cluster_means`, we only apply `scaler_trans.inverse_transform` to the columns it was trained on.
# The one-hot encoded columns should remain as they are (since they are already interpreted as 0 or 1).

# Identify the columns that were originally scaled by `scaler_trans`
original_scaled_cols = ['Transaction_Count', 'Transaction_Amount'] # This should match `features_to_scale_trans`

# Inverse transform only the columns that were scaled
inverse_transformed_numerical_means = scaler_trans.inverse_transform(cluster_means[original_scaled_cols])

# Create a DataFrame for the inverse transformed numerical means
inverse_transformed_df = pd.DataFrame(inverse_transformed_numerical_means, columns=original_scaled_cols)

# Copy the one-hot encoded columns directly from cluster_means as they were not scaled by `scaler_trans`
# and concatenate them
one_hot_cols = [col for col in cluster_means.columns if col not in original_scaled_cols]
inverse_transformed_df[one_hot_cols] = cluster_means[one_hot_cols].values # Direct copy of values

inverse_transformed_df['Cluster'] = range(optimal_n_components)
inverse_transformed_cluster_means = inverse_transformed_df.set_index('Cluster')

print("\nMean feature values for each GMM cluster (inverse transformed to original scale):")
display(inverse_transformed_cluster_means)

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
import pickle

# Define the filename for saving the model
model_filename = 'gmm_best_clustering_model.pkl'

# Save the GMM model to a pickle file
with open(model_filename, 'wb') as file:
    pickle.dump(gmm_model, file)

print(f"Best performing ML model (GMM) successfully saved to '{model_filename}'")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
import pickle
import numpy as np

# Define the filename where the model was saved
model_filename = 'gmm_best_clustering_model.pkl'

# Load the saved GMM model
with open(model_filename, 'rb') as file:
    loaded_gmm_model = pickle.load(file)

print("ML model successfully loaded from pickle file.")

# For sanity check, let's predict on a small sample of the data (e.g., first 5 rows of X_gmm)
# Make sure to drop the 'Cluster_GMM' column if it exists, as it's the target being predicted
unseen_data_sample = X_gmm.drop(columns=['Cluster_GMM'], errors='ignore').head(5)

# Predict the clusters for the unseen data sample
sanity_check_predictions = loaded_gmm_model.predict(unseen_data_sample)

print("\nUnseen data sample (first 5 rows, scaled):")
display(unseen_data_sample)

print("\nPredicted clusters for the unseen data sample:")
print(sanity_check_predictions)

# Also, we can check the probabilities of belonging to each cluster
sanity_check_probabilities = loaded_gmm_model.predict_proba(unseen_data_sample)
print("\nProbabilities of belonging to each cluster for the unseen data sample:")
print(np.round(sanity_check_probabilities, 3))

## ***8. Streamlit Interactive Dashboard***

This section generates the code for a Streamlit dashboard to interactively visualize the PhonePe Pulse data.

**To run this Streamlit app:**
1.  Save the code below into a file named `app.py` (or any other `.py` extension).
2.  Open a new cell in Colab and run the following commands:
    ```bash
    !pip install streamlit
    !streamlit run app.py & npx localtunnel --port 8501
    ```
3.  Click on the localtunnel URL provided in the output to view your dashboard in a new tab.

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

# Database connection
DB_NAME = 'phonepe_pulse.db'

@st.cache_data
def get_data_from_db(table_name):
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    return df

# Load all necessary dataframes
aggregated_transaction_df = get_data_from_db('Aggregated_transaction')
aggregated_user_df = get_data_from_db('Aggregated_user')
aggregated_insurance_df = get_data_from_db('Aggregated_insurance')
map_transaction_df = get_data_from_db('Map_transaction')
map_user_df = get_data_from_db('Map_user')
map_insurance_df = get_data_from_db('Map_insurance')
top_transaction_df = get_data_from_db('Top_transaction')
top_user_df = get_data_from_db('Top_user')
top_insurance_df = get_data_from_db('Top_insurance')

# --- Streamlit Dashboard UI ---
st.set_page_config(layout='wide', page_title='PhonePe Pulse Dashboard')

st.title('📊 PhonePe Pulse Data Analysis')
st.markdown("--- ")

# Sidebar for navigation
st.sidebar.title('Navigation')
options = [
    'Overall Trends',
    'Aggregated Transactions',
    'Aggregated Users',
    'Top Transactions',
    'Top Users',
    'Map Transactions',
    'Map Users',
    'Aggregated Insurance',
    'Top Insurance',
    'Map Insurance'
]
selection = st.sidebar.radio('Go to', options)


# --- Overall Trends Section ---
if selection == 'Overall Trends':
    st.header('Overall Transaction and User Growth Trends')

    # Overall Transaction Amount Trend
    overall_trans_trend = aggregated_transaction_df.groupby(['Year', 'Quarter'])['Transaction_Amount'].sum().reset_index()
    overall_trans_trend['Year_Quarter'] = overall_trans_trend['Year'].astype(str) + '-Q' + overall_trans_trend['Quarter'].astype(str)
    fig_trans_trend = px.line(
        overall_trans_trend,
        x='Year_Quarter',
        y='Transaction_Amount',
        title='Overall Transaction Amount Over Time',
        labels={'Transaction_Amount': 'Total Transaction Amount (INR)', 'Year_Quarter': 'Year and Quarter'}
    )
    st.plotly_chart(fig_trans_trend, use_container_width=True)

    # Overall Registered Users Trend
    overall_user_trend = map_user_df.groupby(['Year', 'Quarter'])['Registered_Users'].sum().reset_index()
    overall_user_trend['Year_Quarter'] = overall_user_trend['Year'].astype(str) + '-Q' + overall_user_trend['Quarter'].astype(str)
    fig_user_trend = px.line(
        overall_user_trend,
        x='Year_Quarter',
        y='Registered_Users',
        title='Overall Registered Users Over Time',
        labels={'Registered_Users': 'Total Registered Users', 'Year_Quarter': 'Year and Quarter'}
    )
    st.plotly_chart(fig_user_trend, use_container_width=True)

# --- Aggregated Transactions Section ---
elif selection == 'Aggregated Transactions':
    st.header('Aggregated Transaction Data Analysis')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year', sorted(aggregated_transaction_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter', sorted(aggregated_transaction_df['Quarter'].unique()))

    filtered_df = aggregated_transaction_df[(aggregated_transaction_df['Year'] == selected_year) & (aggregated_transaction_df['Quarter'] == selected_quarter)]

    st.subheader(f'Transactions by Type in {selected_year} Q{selected_quarter}')
    trans_by_type = filtered_df.groupby('Transaction_Type')['Transaction_Amount'].sum().reset_index()
    fig_pie = px.pie(trans_by_type, values='Transaction_Amount', names='Transaction_Type', title='Transaction Amount by Type')
    st.plotly_chart(fig_pie, use_container_width=True)

    st.subheader(f'Top States by Transaction Amount in {selected_year} Q{selected_quarter}')
    top_states_trans = filtered_df.groupby('State')['Transaction_Amount'].sum().nlargest(10).reset_index()
    fig_bar_states = px.bar(top_states_trans, x='State', y='Transaction_Amount', title='Top 10 States by Transaction Amount')
    st.plotly_chart(fig_bar_states, use_container_width=True)

# --- Aggregated Users Section ---
elif selection == 'Aggregated Users':
    st.header('Aggregated User Data Analysis')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Users)', sorted(aggregated_user_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Users)', sorted(aggregated_user_df['Quarter'].unique()))

    filtered_user_df = aggregated_user_df[(aggregated_user_df['Year'] == selected_year) & (aggregated_user_df['Quarter'] == selected_quarter)]

    st.subheader(f'Users by Brand in {selected_year} Q{selected_quarter}')
    user_by_brand = filtered_user_df.groupby('Brand')['Count'].sum().reset_index()
    fig_brand_bar = px.bar(user_by_brand, x='Brand', y='Count', title='User Count by Phone Brand')
    st.plotly_chart(fig_brand_bar, use_container_width=True)

# --- Top Transactions Section ---
elif selection == 'Top Transactions':
    st.header('Top Transaction Data Analysis (Pincode Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Top Trans)', sorted(top_transaction_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Top Trans)', sorted(top_transaction_df['Quarter'].unique()))

    filtered_top_trans_df = top_transaction_df[(top_transaction_df['Year'] == selected_year) & (top_transaction_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Pincodes by Transaction Amount in {selected_year} Q{selected_quarter}')
    top_pincodes_trans = filtered_top_trans_df.groupby('Pincode')['Transaction_Amount'].sum().nlargest(10).reset_index()
    fig_pincode_bar = px.bar(top_pincodes_trans, x='Pincode', y='Transaction_Amount', title='Top 10 Pincodes by Transaction Amount')
    st.plotly_chart(fig_pincode_bar, use_container_width=True)

# --- Top Users Section ---
elif selection == 'Top Users':
    st.header('Top User Data Analysis (Pincode Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Top User)', sorted(top_user_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Top User)', sorted(top_user_df['Quarter'].unique()))

    filtered_top_user_df = top_user_df[(top_user_df['Year'] == selected_year) & (top_user_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Pincodes by Registered Users in {selected_year} Q{selected_quarter}')
    top_pincodes_users = filtered_top_user_df.groupby('Pincode')['Registered_Users'].sum().nlargest(10).reset_index()
    fig_pincode_users_bar = px.bar(top_pincodes_users, x='Pincode', y='Registered_Users', title='Top 10 Pincodes by Registered Users')
    st.plotly_chart(fig_pincode_users_bar, use_container_width=True)

# --- Map Transactions Section ---
elif selection == 'Map Transactions':
    st.header('Map Transaction Data Analysis (District Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Map Trans)', sorted(map_transaction_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Map Trans)', sorted(map_transaction_df['Quarter'].unique()))

    filtered_map_trans_df = map_transaction_df[(map_transaction_df['Year'] == selected_year) & (map_transaction_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Districts by Transaction Amount in {selected_year} Q{selected_quarter}')
    top_districts_trans = filtered_map_trans_df.groupby('District')['Transaction_Amount'].sum().nlargest(10).reset_index()
    fig_dist_trans_bar = px.bar(top_districts_trans, x='District', y='Transaction_Amount', title='Top 10 Districts by Transaction Amount')
    st.plotly_chart(fig_dist_trans_bar, use_container_width=True)

# --- Map Users Section ---
elif selection == 'Map Users':
    st.header('Map User Data Analysis (District Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Map User)', sorted(map_user_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Map User)', sorted(map_user_df['Quarter'].unique()))

    filtered_map_user_df = map_user_df[(map_user_df['Year'] == selected_year) & (map_user_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Districts by Registered Users in {selected_year} Q{selected_quarter}')
    top_districts_users = filtered_map_user_df.groupby('District')['Registered_Users'].sum().nlargest(10).reset_index()
    fig_dist_users_bar = px.bar(top_districts_users, x='District', y='Registered_Users', title='Top 10 Districts by Registered Users')
    st.plotly_chart(fig_dist_users_bar, use_container_width=True)


# --- Aggregated Insurance Section ---
elif selection == 'Aggregated Insurance':
    st.header('Aggregated Insurance Data Analysis')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Agg Insur)', sorted(aggregated_insurance_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Agg Insur)', sorted(aggregated_insurance_df['Quarter'].unique()))

    filtered_insur_df = aggregated_insurance_df[(aggregated_insurance_df['Year'] == selected_year) & (aggregated_insurance_df['Quarter'] == selected_quarter)]

    st.subheader(f'Insurance Transactions by Type in {selected_year} Q{selected_quarter}')
    insur_by_type = filtered_insur_df.groupby('Transaction_Type')['Transaction_Amount'].sum().reset_index()
    fig_insur_pie = px.pie(insur_by_type, values='Transaction_Amount', names='Transaction_Type', title='Insurance Transaction Amount by Type')
    st.plotly_chart(fig_insur_pie, use_container_width=True)


# --- Top Insurance Section ---
elif selection == 'Top Insurance':
    st.header('Top Insurance Data Analysis (Pincode Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Top Insur)', sorted(top_insurance_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Top Insur)', sorted(top_insurance_df['Quarter'].unique()))

    filtered_top_insur_df = top_insurance_df[(top_insurance_df['Year'] == selected_year) & (top_insurance_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Pincodes by Insurance Transaction Amount in {selected_year} Q{selected_quarter}')
    top_pincodes_insur = filtered_top_insur_df.groupby('Pincode')['Transaction_Amount'].sum().nlargest(10).reset_index()
    fig_pincode_insur_bar = px.bar(top_pincodes_insur, x='Pincode', y='Transaction_Amount', title='Top 10 Pincodes by Insurance Transaction Amount')
    st.plotly_chart(fig_pincode_insur_bar, use_container_width=True)

# --- Map Insurance Section ---
elif selection == 'Map Insurance':
    st.header('Map Insurance Data Analysis (District Level)')

    col1, col2 = st.columns(2)
    with col1:
        selected_year = st.selectbox('Select Year (Map Insur)', sorted(map_insurance_df['Year'].unique()))
    with col2:
        selected_quarter = st.selectbox('Select Quarter (Map Insur)', sorted(map_insurance_df['Quarter'].unique()))

    filtered_map_insur_df = map_insurance_df[(map_insurance_df['Year'] == selected_year) & (map_insurance_df['Quarter'] == selected_quarter)]

    st.subheader(f'Top 10 Districts by Insurance Transaction Amount in {selected_year} Q{selected_quarter}')
    top_districts_insur = filtered_map_insur_df.groupby('District')['Transaction_Amount'].sum().nlargest(10).reset_index()
    fig_dist_insur_bar = px.bar(top_districts_insur, x='District', y='Transaction_Amount', title='Top 10 Districts by Insurance Transaction Amount')
    st.plotly_chart(fig_dist_insur_bar, use_container_width=True)


### Run the Streamlit Dashboard

Execute the cell below to install Streamlit and localtunnel, then launch your dashboard. Click the public URL provided in the output to access it.

In [ ]:
!pip install streamlit
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project undertook an in-depth analysis of PhonePe transaction data, creating an end-to-end data analytics and business intelligence platform. We successfully extracted, transformed, and loaded raw JSON data into structured DataFrames, laying the groundwork for comprehensive analysis.

### Key Findings from Exploratory Data Analysis (EDA):
*   **Consistent Growth:** The overall transaction amount, registered users, and app opens demonstrated strong, consistent year-over-year growth, indicating increasing adoption and market penetration of PhonePe.
*   **Dominance of P2P Payments:** 'Peer-to-peer payments' consistently represented the largest proportion of total transaction amounts, followed by 'Merchant payments,' which also showed significant growth.
*   **Geographical Concentration:** User registration and transaction amounts were concentrated in key states like Maharashtra, Karnataka, and Uttar Pradesh, highlighting major digital payment hubs.
*   **Device Preferences:** Insights into dominant phone brands among users can inform app optimization and targeted marketing.
*   **Correlation:** A strong positive correlation was observed between `Transaction_Count` and `Transaction_Amount`, as well as a positive trend with `Year`, reinforcing overall growth.

### Hypothesis Testing:
Through statistical tests, we confirmed three key hypotheses:
1.  **Transaction Growth:** The total transaction amount on PhonePe has significantly increased year-over-year.
2.  **User Engagement Growth:** The number of registered users on PhonePe has consistently grown each year.
3.  **Transaction Type Dominance:** 'Peer-to-peer payments' consistently represent more than 50% of the total transaction amount.

### Data Preprocessing and Feature Engineering:
*   Missing values were meticulously handled by dropping minimal rows with null `Pincode` entries, ensuring data integrity.
*   Numerical features were scaled using `StandardScaler` to prepare the data for distance-based clustering algorithms, preventing features with larger magnitudes from dominating.
*   Categorical features like `Transaction_Type` and `Brand` were one-hot encoded to be suitable for machine learning models.

### Machine Learning Model Implementation (Unsupervised Learning):
We implemented and evaluated three clustering models: K-Means, Agglomerative Clustering, and Gaussian Mixture Models (GMM).
*   **K-Means:** Provided a baseline for partitioning data into `k=4` clusters, chosen using the Elbow method.
*   **Agglomerative Clustering:** Explored hierarchical relationships, and with `n_clusters=4` and 'complete' linkage, it showed improved performance over K-Means.
*   **Gaussian Mixture Model (GMM):** Emerged as the best-performing model with **10 components** and a **'tied' covariance type**, achieving the highest Silhouette Score (0.8701) and lowest Davies-Bouldin Index (0.4314). Its probabilistic approach and flexibility in modeling cluster shapes provided a more nuanced and accurate segmentation.

### Business Impact of GMM:
The chosen GMM offers significant business value:
*   **Refined Customer Segmentation:** The high Silhouette Score and low Davies-Bouldin Index instill confidence in the distinctness of the identified segments, enabling highly targeted marketing campaigns, personalized product recommendations, and tailored service offerings.
*   **Nuanced Insights:** GMM's probabilistic assignments and ability to model complex cluster shapes allow PhonePe to uncover more realistic and granular user behaviors and transaction patterns.
*   **Enhanced Decision Support:** By understanding the unique characteristics of each of the 10 clusters (derived from feature means), PhonePe can make more informed strategic decisions regarding product development, resource allocation, risk management, and market expansion.

### Conclusion:
This project successfully built a robust data analytics pipeline, from data extraction to advanced machine learning-driven segmentation. The insights gained provide PhonePe with a powerful toolset for understanding its dynamic digital payments ecosystem, fostering continued growth, and making data-driven strategic decisions. The final GMM model, capable of reliable segmentation, is now ready for deployment to support real-time business intelligence.